# 📡 Wi-Fi: от основ до настройки и использования

## Обучающий Colab-ноутбук для начинающих

Добро пожаловать в интерактивный курс по Wi-Fi. Курс построен от фундамента к практике: сначала разберёмся, что такое радиоволны вообще, затем — как Wi-Fi использует их для передачи данных, и в конце — как всё это настраивать и диагностировать в реальной жизни.

---

### 🎯 Что вы изучите

После прохождения ноутбука вы будете:

- **Понимать** как радиоволна передаёт данные и почему Wi-Fi работает именно так.
- **Различать** стандарты Wi-Fi 4, 5, 6, 6E, 7 и их характеристики.
- **Оценивать** безопасность сети и выбирать стойкие пароли.
- **Настраивать** роутер: каналы, мощность, шифрование, DHCP, роуминг.
- **Диагностировать** медленный интернет и находить причину за 15 минут.

### 📚 Структура курса

| № | Тема | Ключевые понятия |
|---|---|---|
| 1 | Радиоволны: что это и какими бывают | Электромагнитный спектр, λ=c/f, диапазоны от ELF до EHF |
| 2 | Wi-Fi как применение радиоволн | Модуляция, AP, STA, антенны, FSPL |
| 3 | Стандарты Wi-Fi и частоты | 802.11a/b/g/n/ac/ax/be, 2.4/5/6 ГГц, каналы |
| 4 | Безопасность Wi-Fi | WPA2, WPA3, 4-way handshake, SAE, энтропия |
| 5 | Настройка сети | Каналы, TX Power, DHCP, Band Steering, роуминг |
| 6 | Анализ и оптимизация | RSSI, SNR, Channel Utilization, retry rate, MCS, диагностика |

### 🛠️ Как работать с ноутбуком

1. Запустите по очереди все ячейки сверху вниз.
2. В каждой теме читайте объяснения и экспериментируйте с параметрами.
3. Отвечайте на мини-проверки и тесты, разворачивая ответы для самопроверки.
4. После каждой темы проходите чек-лист — если все пункты согласны, идёте дальше.

> 💡 **Совет:** не спешите. Лучше пройти 1 тему за час и понять, чем 6 тем за час и не запомнить ничего.

## 🔧 Подготовка окружения

В этом ноутбуке используются стандартные библиотеки Python + несколько дополнительных. Запустите ячейку ниже, чтобы установить всё необходимое.

In [ ]:
# Установка необходимых библиотек
# В Google Colab большинство уже предустановлены, но для надёжности выполним явно
!pip install -q numpy pandas matplotlib cryptography

print('=== Проверка окружения ===')
import sys, platform
print(f'Python:    {sys.version.split()[0]}')
print(f'ОС:        {platform.system()} {platform.release()}')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from cryptography import __version__ as crypto_ver

print(f'numpy:        {np.__version__}')
print(f'pandas:       {pd.__version__}')
print(f'matplotlib:   {matplotlib.__version__}')
print(f'cryptography: {crypto_ver}')

# Настройка шрифтов для русского языка в matplotlib
import matplotlib.font_manager as fm
import os
_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf',
    '/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]
for _p in _FONT_CANDIDATES:
    if os.path.exists(_p):
        try: fm.fontManager.addfont(_p)
        except Exception: pass
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Liberation Sans', 'FreeSans']
plt.rcParams['axes.unicode_minus'] = False

print()
print('✅ Окружение готово! Можно приступать к Теме 1.')

## 📐 Соглашения и обозначения

В этом ноутбуке используются следующие обозначения:

| Обозначение | Значение |
|---|---|
| 📘 | Заголовок темы |
| 🎯 | Краткий вывод |
| 🔬 | Мини-лабораторная |
| 🧪 | Эксперимент |
| ✅ | Мини-проверка или чек-лист |
| ⚠️ | Частые ошибки |
| 📝 | Тестовые задания |
| 🧠 | Проверка понимания |
| 🔗 | Связь между темами |
| 😄 | Юмор (тема-related, без потери точности) |

Все единицы измерения — СИ, кроме случаев, где традиционно используются другие (дБм для мощности, Мбит/с для скорости, ГГц для частоты).

---

# 📘 Тема 1. Радиоволны: что это и какими бывают

## Пять вопросов темы

| Вопрос | Ответ |
|---|---|
| **Что это?** | Радиоволна — это электромагнитная волна с частотой от 3 кГц до 300 ГГц, распространяющаяся в пространстве со скоростью света и переносящая энергию (и информацию). |
| **Зачем существует?** | Радиоволны существуют потому, что изменяющееся электрическое поле порождает магнитное, а изменяющееся магнитное — электрическое. Этот «электромагнитный танец» и распространяется в пространстве без носителя. |
| **Как работает?** | Антенна передатчика создаёт переменный ток, который порождает электромагнитное поле. Поле отрывается от антенны и распространяется в пространстве со скоростью света (~3·10⁸ м/с). Антенна приёмника ловит это поле и преобразует обратно в ток. |
| **Почему именно так?** | Потому что электромагнитное взаимодействие — одно из четырёх фундаментальных взаимодействий физики. У него бесконечный радиус действия и не нужен носитель (в отличие от звука, которому нужна среда). |
| **Где применяется?** | Радио- и телетрансляция, мобильная связь, Wi-Fi, Bluetooth, GPS, радары, спутниковая связь, микроволновки, рентген — всё это электромагнитные волны разных частот. |

---

## Шаг 1. Постановка проблемы

Представьте, что вы хотите передать сигнал другу, который стоит в 100 метрах от вас. Варианты:

- **Крикнуть** — звук дойдёт, но только если нет шума и ветер попутный. Скорость звука ~340 м/с, нужен воздух.
- **Послать письмо голубиной почтой** — надёжно, но медленно (минуты-часы).
- **Сигнал фонарём** — быстрее, но только в прямой видимости и ночью.
- **Передать радиосигнал** — мгновенно, сквозь стены, ночью и днём, на километры.

Звук — это механическая волна: ей нужна среда (воздух, вода, металл). В космосе звука нет. А вот радиоволна — электромагнитная, ей среда не нужна. Она распространяется и в воздухе, и в вакууме, и сквозь неметаллические стены.

**Проблема:** как передавать информацию без физического носителя, мгновенно, на большие расстояния и в любых условиях?

**Решение:** использовать электромагнитные волны, в частности — радиоволны. Это и есть физическая основа всех беспроводных технологий: радио, ТВ, мобильной связи, спутников, Wi-Fi.

## Шаг 2. Практический пример

Давайте «потрогаем» радиоволны программно. Посмотрим, где в электромагнитном спектре находятся разные знакомые нам технологии — от радиостанций до рентгена.

In [ ]:
# Электромагнитный спектр: от длинных радиоволн до гамма-излучения
import pandas as pd

spectrum = [
    # (название, диапазон частот, длина волны, примеры применения)
    ('Низкие частоты (ELF/VLF)', '3 Гц – 30 кГц', '10000–100 км', 'Связь с подлодками, геофизика'),
    ('Длинные волны (LW)', '30–300 кГц', '10–1 км', 'Радиовещание, навигация'),
    ('Средние волны (MW)', '300–3000 кГц', '1000–100 м', 'АМ-радио ( средневолновые станции)'),
    ('Короткие волны (SW)', '3–30 МГц', '100–10 м', 'Международное радио, радиолюбители'),
    ('Ультракороткие волны (VHF)', '30–300 МГц', '10–1 м', 'FM-радио, ТВ, авиасвязь'),
    ('Дециметровые (UHF)', '300–3000 МГц', '100–10 см', 'ТВ, GSM, GPS, Wi-Fi 2.4 ГГц, Bluetooth'),
    ('Сантиметровые (SHF)', '3–30 ГГц', '10–1 см', 'Wi-Fi 5/6 ГГц, радары, спутники, 5G'),
    ('Миллиметровые (EHF)', '30–300 ГГц', '10–1 мм', 'Wi-Fi 60 ГГц, 5G mmWave, астрономия'),
    ('Терагерцовые', '300 ГГц – 3 ТГц', '1 мм – 100 мкм', 'Сквозная диагностика, наука'),
    ('Инфракрасные', '3–430 ТГц', '100 мкм – 700 нм', 'Пульты ДУ, тепловизоры, оптоволокно'),
    ('Видимый свет', '430–770 ТГц', '700–400 нм', 'Зрение, оптическая связь'),
    ('Ультрафиолет', '770 ТГц – 30 ПГц', '400–10 нм', 'Дезинфекция, люминесценция'),
    ('Рентген', '30 ПГц – 30 ЭГц', '10 нм – 10 пм', 'Медицина, контроль материалов'),
    ('Гамма-излучение', '> 30 ЭГц', '< 10 пм', 'Ядерная физика, стерилизация'),
]

df = pd.DataFrame(spectrum, columns=['Диапазон', 'Частоты', 'Длина волны', 'Применение'])
df

**Разбор результата.** Таблица показывает весь электромагнитный спектр. Радиоволны — это только нижняя часть спектра: от 3 Гц до 300 ГГц. Видимый свет — это тоже электромагнитные волны, но с частотой в 1 миллион раз выше. Рентген — ещё выше. Между ними — инфракрасное излучение (тепло) и ультрафиолет (загар).

Wi-Fi работает в дециметровом и сантиметровом диапазонах (2,4–6 ГГц), где волны имеют длину 5–12 см. Это позволяет им проникать сквозь стены и не требовать прямой видимости.

## Шаг 3. Интуитивное объяснение

Если бросить камень в пруд — пойдут круги. Это **механическая** волна: вода колеблется вверх-вниз, и колебание распространяется от центра.

Радиоволна — похожий процесс, но в роли «воды» выступает само пространство. Электрическое и магнитное поля колеблются синхронно, перпендикулярно друг другу и направлению движения. Эти колебания распространяются от антенны во все стороны со скоростью света.

Главное отличие от воды: радиоволне **не нужна среда**. Она идёт и в воздухе, и в вакууме. Именно поэтому мы можем общаться с марсоходом по радио, хотя между нами — пустота космоса.

## Шаг 4. Жизненная аналогия

Представьте **большую пружину** («Слинки»), растянутую между двумя людьми. Один резко дёргает свой конец вверх-вниз — по пружине бежит волна, и через секунду другой человек видит, что его конец тоже качнулся.

Радиоволна — это «Слинки», только вместо пружины — электромагнитное поле, пронизывающее всё пространство. Антенна передатчика «дёргает» поле — и волна бежит во все стороны. Антенна приёмника ощущает качание поля и преобразует его обратно в электрический сигнал.

Частота, с которой вы дёргаете пружину — это частота радиоволны. Расстояние между гребнями — это длина волны. Чем быстрее дёргаете, тем короче волна.

## Шаг 5. Техническое объяснение

Радиоволна описывается тремя основными параметрами:

| Параметр | Обозначение | Единица | Смысл |
|---|---|---|---|
| **Частота** | f | Герц (Гц) | Сколько колебаний в секунду |
| **Длина волны** | λ (лямбда) | метр (м) | Расстояние между соседними гребнями |
| **Скорость** | c | м/с | Скорость света в среде (в вакууме c ≈ 3·10⁸ м/с) |

Эти параметры связаны фундаментальной формулой:

## **λ = c / f**

Чем выше частота, тем короче длина волны, и наоборот. Например:
- FM-радио на 100 МГц: λ = 3·10⁸ / 10⁸ = **3 метра**
- Wi-Fi на 2,4 ГГц: λ = 3·10⁸ / 2,4·10⁹ = **12,5 см**
- Wi-Fi на 6 ГГц: λ = 3·10⁸ / 6·10⁹ = **5 см**
- Видимый свет (зелёный, 540 ТГц): λ = 3·10⁸ / 5,4·10¹⁴ = **555 нанометров**

От длины волны зависят физические свойства:
- **Длинные волны** (> 100 м) огибают препятствия, отражаются от ионосферы, обходят Землю.
- **Короткие волны** (10–100 м) отражаются от ионосферы — связь между континентами.
- **УКВ** (1–10 м) распространяются только в прямой видимости — FM-радио, ТВ.
- **Микроволны** (< 1 м) почти не огибают препятствия, но несут много данных — Wi-Fi, спутники.

## Шаг 6. Внутреннее устройство

Как создаётся радиоволна:

```mermaid
flowchart LR
    A[Источник тока<br/>переменный] --> B[Антенна<br/>передатчика]
    B -->|переменный ток порождает<br/>переменное E-поле| E1((Электрическое<br/>поле))
    E1 -->|изменяющееся E-поле<br/>порождает B-поле| M1((Магнитное<br/>поле))
    M1 -->|изменяющееся B-поле<br/>порождает E-поле| E2((Электрическое<br/>поле))
    E2 --> M2((Магнитное<br/>поле))
    M2 -.распространение<br/>в пространстве.-> R[Антенна<br/>приёмника]
    R --> O[Восстановленный<br/>ток в приёмнике]
```

Это явление описано **уравнениями Максвелла** (1865 год). Они показывают, что:
1. Переменный ток в антенне создаёт переменное электрическое поле.
2. Переменное электрическое поле создаёт переменное магнитное поле (закон Фарадея).
3. Переменное магнитное поле создаёт переменное электрическое поле (закон Ампера-Максвелла).
4. Эти поля «поддерживают» друг друга и распространяются в пространстве как волна.

**Классификация радиоволн по длине** (официальная ITU-R):

| Диапазон | Сокращение | Частоты | Длины волн | Применение |
|---|---|---|---|---|
| Низкие/очень низкие | ELF/VLF | 3 Гц – 30 кГц | 10000–100 км | Подводная связь |
| Длинные | LF | 30–300 кГц | 10–1 км | Радиовещание LW |
| Средние | MF | 300–3000 кГц | 1 км – 100 м | АМ-радио |
| Короткие | HF | 3–30 МГц | 100–10 м | Международное радио |
| Ультракороткие метровые | VHF | 30–300 МГц | 10–1 м | FM-радио, ТВ |
| Дециметровые | UHF | 300–3000 МГц | 100–10 см | ТВ, GSM, GPS, Wi-Fi 2.4 ГГц |
| Сантиметровые | SHF | 3–30 ГГц | 10–1 см | Wi-Fi 5 ГГц, радары, 5G |
| Миллиметровые | EHF | 30–300 ГГц | 10–1 мм | Wi-Fi 60 ГГц, 5G mmWave |

## Шаг 7. Визуальная схема

Электромагнитный спектр (логарифмическая шкала):

```mermaid
flowchart LR
    subgraph low[Низкие частоты - длинные волны]
        ELF[ELF/VLF<br/>3 Гц-30 кГц] --> LF[LF<br/>длинные]
        LF --> MF[MF<br/>средние АМ]
        MF --> HF[HF<br/>короткие]
    end
    subgraph mid[Радиоволны]
        HF --> VHF[VHF<br/>УКВ FM/ТВ]
        VHF --> UHF[UHF<br/>Wi-Fi 2.4 ГГц, GSM, GPS]
        UHF --> SHF[SHF<br/>Wi-Fi 5/6 ГГц, 5G, радары]
        SHF --> EHF[EHF<br/>Wi-Fi 60 ГГц, mmWave]
    end
    subgraph high[Оптика и выше]
        EHF --> IR[ИК<br/>пульты, тепло]
        IR --> VIS[Видимый свет<br/>700-400 нм]
        VIS --> UV[УФ<br/>загар]
        UV --> XR[Рентген<br/>медицина]
        XR --> GM[Гамма<br/>ядерная физика]
    end
    style low fill:#dbeafe
    style mid fill:#fef3c7
    style high fill:#fee2e2
```

Все эти диапазоны — **одна и та же физика** электромагнитных волн. Разница только в частоте (и, следовательно, в длине волны). Wi-Fi — это просто радиоволны определённого диапазона (UHF + SHF).

## Шаг 8. Рабочий пример

Визуализируем электромагнитный спектр и покажем, где именно находятся знакомые нам технологии.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Поиск шрифта с поддержкой кириллицы по нескольким известным путям
_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf',
    '/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]
for _p in _FONT_CANDIDATES:
    if os.path.exists(_p):
        try:
            fm.fontManager.addfont(_p)
        except Exception:
            pass
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Liberation Sans', 'FreeSans']
plt.rcParams['axes.unicode_minus'] = False

# Параметры известных технологий: (название, частота в Гц, цвет)
technologies = [
    ('AM-радио',         1e6,       '#3b82f6'),
    ('FM-радио',         1e8,       '#06b6d4'),
    ('Телевидение',      5e8,       '#10b981'),
    ('GSM 900',          9e8,       '#22c55e'),
    ('GPS',              1.575e9,   '#84cc16'),
    ('Wi-Fi 2.4 ГГц',    2.4e9,     '#eab308'),
    ('Bluetooth',        2.45e9,    '#f59e0b'),
    ('Wi-Fi 5 ГГц',      5e9,       '#f97316'),
    ('Wi-Fi 6 ГГц',      6e9,       '#ef4444'),
    ('Wi-Fi 60 ГГц',     6e10,      '#dc2626'),
    ('5G mmWave',        3e10,      '#b91c1c'),
    ('ИК-пульт',         3e14,      '#7c3aed'),
    ('Видимый свет',     5.4e14,    '#a855f7'),
    ('УФ',               1e15,      '#6366f1'),
    ('Рентген',          1e18,      '#3730a3'),
]

fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)

# Закрашиваем диапазоны спектра
bands = [
    (3e3, 3e8, '#dbeafe', 'Радиоволны'),
    (3e8, 4.3e14, '#fef3c7', 'Микроволны + ИК'),
    (4.3e14, 7.7e14, '#fce7f3', 'Видимый свет'),
    (7.7e14, 3e16, '#fee2e2', 'УФ'),
    (3e16, 3e19, '#ede9fe', 'Рентген'),
    (3e19, 1e22, '#f3e8ff', 'Гамма'),
]
for f_min, f_max, color, label in bands:
    ax.axvspan(f_min, f_max, alpha=0.4, color=color, label=label)

# Точки технологий
for name, freq, color in technologies:
    ax.scatter(freq, 1, s=180, color=color, edgecolor='black', linewidth=1, zorder=5)
    # Подпись с переносом
    label = name.replace(' ', '\n', 1) if len(name) > 10 else name
    ax.annotate(name, (freq, 1), xytext=(0, 20 if technologies.index((name, freq, color)) % 2 == 0 else -35),
                textcoords='offset points', ha='center', fontsize=8, fontweight='bold',
                rotation=45)

ax.set_xscale('log')
ax.set_xlim(1e3, 1e22)
ax.set_ylim(-0.5, 2)
ax.set_yticks([])
ax.set_xlabel('Частота, Гц (логарифмическая шкала)', fontsize=11)
ax.set_title('Электромагнитный спектр: где живут известные технологии',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, ncol=3)
ax.grid(True, alpha=0.3, axis='x')
plt.show()

**Разбор результата.** На графике видно: Wi-Fi занимает крошечную часть спектра — узкую полоску между 2,4 и 6 ГГц. Соседние технологии — Bluetooth (2,45 ГГц), GPS (1,575 ГГц), 5G mmWave (28–39 ГГц). Все они сосуществуют в эфире, не мешая друг другу, потому что работают на разных частотах.

Чем правее на графике (выше частота), тем короче длина волны и тем больше данных можно передать, но тем хуже сигнал проникает сквозь препятствия. Видимый свет имеет длину волны ~500 нм — это миллионы раз короче, чем у Wi-Fi, и поэтому свет не проходит сквозь стены.

## Шаг 9. Практический эксперимент

Рассчитаем и сравним длины волн для разных технологий. Меняйте частоты и наблюдайте, как меняется длина волны.

In [ ]:
import math

C = 3e8  # скорость света в вакууме, м/с

def wavelength(freq_hz: float) -> float:
    """Возвращает длину волны в метрах."""
    return C / freq_hz

def format_length(m: float) -> str:
    if m >= 1000: return f'{m/1000:.1f} км'
    if m >= 1:    return f'{m:.2f} м'
    if m >= 0.01: return f'{m*100:.2f} см'
    if m >= 1e-6: return f'{m*1000:.2f} мм'
    if m >= 1e-9: return f'{m*1e9:.2f} нм'
    return f'{m:.2e} м'

# === ЭКСПЕРИМЕНТ: меняйте список частот ===
test_freqs = [
    ('AM-радио',         1e6),
    ('FM-радио',         1e8),
    ('Wi-Fi 2.4 ГГц',    2.4e9),
    ('Wi-Fi 5 ГГц',      5e9),
    ('Wi-Fi 6 ГГц',      6e9),
    ('Wi-Fi 60 ГГц',     60e9),
    ('5G mmWave (28 ГГц)', 28e9),
    ('ИК-пульт (3 ТГц)', 3e12),
    ('Зелёный свет',     5.4e14),
    ('Рентген',          1e18),
]

print(f"{'Технология':<25}{'Частота':<20}{'Длина волны':<20}")
print('-' * 65)
for name, f in test_freqs:
    lam = wavelength(f)
    # Форматируем частоту
    if f < 1e3: f_str = f'{f} Гц'
    elif f < 1e6: f_str = f'{f/1e3:.1f} кГц'
    elif f < 1e9: f_str = f'{f/1e6:.2f} МГц'
    elif f < 1e12: f_str = f'{f/1e9:.2f} ГГц'
    elif f < 1e15: f_str = f'{f/1e12:.2f} ТГц'
    else: f_str = f'{f/1e15:.2f} ПГц'
    print(f'{name:<25}{f_str:<20}{format_length(lam):<20}')

print()
print('Вывод: при росте частоты в 1000 раз длина волны падает в 1000 раз.')
print('AM-радио: волна 300 м (огибает здания).')
print('Wi-Fi 6 ГГц: волна 5 см (почти не огибает даже человека).')
print('Рентген: волна 0.3 нм (проходит сквозь мягкие ткани, задерживается костями).')

**Что наблюдаем.** При росте частоты в 1000 раз (1 МГц → 1 ГГц) длина волны падает в 1000 раз (300 м → 30 см). Это обратная пропорциональность из формулы λ = c / f. Длина волны напрямую определяет физические свойства: длинные волны огибают препятствия, короткие — отражаются или поглощаются.

## 🧠 Проверка понимания

**1. (Объяснение)** Почему радиоволна может распространяться в вакууме, а звук — нет?

**2. (Прогноз)** Если удвоить частоту радиоволны, что произойдёт с длиной волны?

**3. (Объяснение)** Почему FM-радио (волна ~3 м) слышно дальше за горизонтом, чем Wi-Fi (волна ~12 см) при той же мощности?

---

## ⚠️ Частые ошибки

### Ошибка 1

**Неправильное рассуждение:** «Радиоволны и звук — это одно и то же, просто на разных частотах».

**Причина ошибки:** Путают механические и электромагнитные волны.

**Правильное объяснение:** Звук — это механическая волна (колебания давления в среде). Радиоволна — электромагнитная (колебания E и B полей). Звук не распространяется в вакууме, радиоволна — да. Это разная физика.

### Ошибка 2

**Неправильное рассуждение:** «Чем выше частота, тем дальше бьёт сигнал».

**Причина ошибки:** Логика «сильнее = дальше».

**Правильное объяснение:** Наоборот: чем выше частота (короче волна), тем хуже сигнал огибает препятствия и тем сильнее поглощается. Длинные волны (LW, AM) огибают Землю, миллиметровые (Wi-Fi 60 ГГц) — останавливаются стеной.

### Ошибка 3

**Неправильное рассуждение:** «Радиоволны — это что-то искусственное, придуманное людьми».

**Причина ошибки:** Знают только о радиоприёмниках.

**Правильное объяснение:** Радиоволны существуют в природе: Солнце, звёзды, молнии, магнитное поле Земли — всё излучает электромагнитные волны. Люди научились их генерировать и принимать искусственно, но не изобрели.

### Ошибка 4

**Неправильное рассуждение:** «Видимый свет и радиоволны — разные явления».

**Причина ошибки:** Воспринимают их разными органами чувств.

**Правильное объяснение:** Это одна и та же физика: электромагнитные волны. Видимый свет — радиоволны с частотой 430–770 ТГц. Рентген — радиоволны с частотой 10¹⁸ Гц. Разница только в частоте (и, соответственно, в энергии фотонов).

---

## 🎯 Краткий вывод

Радиоволны — это электромагнитные волны с частотой от 3 Гц до 300 ГГц. Они создаются переменным током в антенне и распространяются со скоростью света, не требуя среды. Длина волны обратно пропорциональна частоте: λ = c / f. Радиоволны делятся на диапазоны от ELF (подводная связь) до EHF (60 ГГц Wi-Fi). Чем выше частота, тем короче волна и тем больше данных можно передать, но тем хуже проникновение сквозь препятствия. Радиоволны — фундамент всех беспроводных технологий: радио, ТВ, мобильной связи, GPS, спутников, и, конечно, Wi-Fi.

> 😄 Радиоволна — это как кружок по воде, только вместо воды — само пространство, а вместо камушка — антенна. Только круги расходятся со скоростью света, и остановить их может только клетка Фарадея.

---


## 🔗 Связь между темами

### Какие знания были получены

- Понятие электромагнитной волны и её связь с электрическим и магнитным полями
- Диапазоны радиоволн: от ELF до EHF, и их применения
- Формулу λ = c / f и физику распространения волн
- Понимание, что Wi-Fi — это радиоволны диапазонов UHF и SHF

### Какие знания понадобятся далее

- Как именно радиоволны используются для передачи данных
- Что такое модуляция и как биты превращаются в радиосигнал
- Как устроена Wi-Fi-система: точка доступа, клиенты, антенны
- Формула FSPL — потери сигнала в свободном пространстве

**Переход к следующей теме:** Мы поняли, что такое радиоволна и какие бывают её виды. Но радиоволна сама по себе — это просто «несущая». В следующей теме мы разберём, как именно радиоволна используется для передачи данных: что такое модуляция, как Wi-Fi-система устроена, и почему именно частоты 2,4 / 5 / 6 ГГц были выбраны для беспроводных сетей.

---

## ✅ Мини-проверка

**Вопрос 1** (объяснить причину):

Почему радиоволны могут распространяться в вакууме, а звук — нет?

**Вопрос 2** (предсказать результат):

Если удвоить частоту с 1 МГц до 2 МГц, что произойдёт с длиной волны?

**Вопрос 3** (выбрать правильный вариант):

В каком диапазоне работает Wi-Fi 5 ГГц? (a) LF (b) MF (c) UHF (d) SHF

**Вопрос 4** (найти ошибку):

В коде `wavelength = 3e8 / 100` нашли длину волны для «FM-радио». Что не так?

---

## 🔬 Мини-лабораторная

**Цель:** Научиться рассчитывать длину волны для разных технологий и сравнивать их физические свойства.

### Пошаговая инструкция

1. Откройте новую ячейку.
2. Используйте формулу λ = c / f, где c = 3·10⁸ м/с.
3. Рассчитайте длины волн для: AM-радио (1 МГц), FM-радио (100 МГц), Wi-Fi 2.4 ГГц, Wi-Fi 5 ГГц, Wi-Fi 6 ГГц, Wi-Fi 60 ГГц, 5G mmWave (28 ГГц), ИК (3 ТГц), зелёного света (540 ТГц).
4. Выведите результаты в таблицу через pandas.
5. Постройте логарифмический график: по оси X — частота, по оси Y — длина волны.
6. Найдите, при какой частоте длина волны становится меньше 1 мм.

**Ожидаемый результат:** Таблица с 9 значениями и логарифмический график, где видно обратную пропорциональность λ и f.

**Объяснение результата:** Эта лабораторная закрепляет фундаментальное понимание: частота и длина волны — две стороны одной медали. Выбор частоты для технологии определяет её физику: дальность, проникновение, скорость передачи.

---

## 🧪 Эксперимент

**Задание:** В коде Шага 9 поменяйте частоты и проверьте, выполняется ли правило «удвоение частоты = деление длины волны пополам».

Изменяемые параметры:

- Возьмите 2 МГц, 4 МГц, 8 МГц, 16 МГц
- Затем — 2.4 ГГц, 4.8 ГГц, 9.6 ГГц

**Что наблюдаем:** Каждое удвоение частоты должно давать уменьшение длины волны ровно в 2 раза. Это прямое следствие формулы λ = c / f.

---

## 📝 Тестовые задания с вариантами ответов

> Выберите один правильный вариант в каждом задании. Ответы — в конце блока.

### Задание 1

Что такое радиоволна?

- [ ] **a)** Механическое колебание воздуха
- [ ] **b)** Электромагнитная волна с частотой 3 Гц – 300 ГГц
- [ ] **c)** Поток электронов в проводе
- [ ] **d)** Звуковая волна с высокой частотой

### Задание 2

Какая формула связывает длину волны и частоту?

- [ ] **a)** λ = c · f
- [ ] **b)** λ = c / f
- [ ] **c)** λ = f / c
- [ ] **d)** λ = c + f

### Задание 3

Почему длинные радиоволны огибают препятствия лучше, чем короткие?

- [ ] **a)** У них больше энергия
- [ ] **b)** Длина волны сравнима с размером препятствия — дифракция
- [ ] **c)** Они отражаются от ионосферы
- [ ] **d)** Они имеют большую мощность

### Задание 4

В каком диапазоне работает Wi-Fi 2.4 ГГц?

- [ ] **a)** HF (короткие волны, 3–30 МГц)
- [ ] **b)** VHF (УКВ, 30–300 МГц)
- [ ] **c)** UHF (дециметровые, 300–3000 МГц)
- [ ] **d)** SHF (сантиметровые, 3–30 ГГц)

### Задание 5

Какое утверждение про видимый свет верное?

- [ ] **a)** Это совершенно другое явление, не связанное с радиоволнами
- [ ] **b)** Это электромагнитные волны с частотой 430–770 ТГц
- [ ] **c)** Это механические волны, как звук
- [ ] **d)** Это поток электронов

<details><summary><b>🔑 Ответы и пояснения (нажмите, чтобы развернуть)</b></summary>

**Задание 1:** правильный ответ — **b)**. Радиоволна — электромагнитная волна (колебания E и B полей) в диапазоне 3 Гц – 300 ГГц. Не требует среды для распространения, в отличие от звука.

**Задание 2:** правильный ответ — **b)**. λ = c / f, где c — скорость света (~3·10⁸ м/с), f — частота. Чем выше частота, тем короче волна.

**Задание 3:** правильный ответ — **b)**. Волна огибает препятствие, если её длина сравнима с размером препятствия или больше (явление дифракции). У AM-радио волна 300 м — она легко огибает здания. У Wi-Fi волна 12 см — она отражается от стены.

**Задание 4:** правильный ответ — **c)**. 2.4 ГГц = 2400 МГц, попадает в UHF (300–3000 МГц). Wi-Fi 5 ГГц — уже в SHF (3–30 ГГц).

**Задание 5:** правильный ответ — **b)**. Видимый свет — те же электромагнитные волны, что и радиоволны, но с частотой в миллион раз выше. Глаз просто чувствителен к этому узкому диапазону спектра.

</details>

---

## ✅ Чек-лист темы

- Я понимаю, что радиоволна — это электромагнитная волна, а не звук.
- Я понимаю, что радиоволны могут распространяться в вакууме.
- Я умею рассчитывать длину волны по формуле λ = c / f.
- Я умею классифицировать радиоволны по диапазонам (ELF, VHF, UHF, SHF и т.д.).
- Я могу объяснить, почему длинные волны огибают препятствия лучше коротких.
- Я могу объяснить, что видимый свет и радиоволны — это одна и та же физика.

---

# 📘 Тема 2. Wi-Fi как применение радиоволн

## Пять вопросов темы

| Вопрос | Ответ |
|---|---|
| **Что это?** | Wi-Fi — это набор стандартов (IEEE 802.11), использующих радиоволны на частотах 2,4 / 5 / 6 ГГц для беспроводной передачи данных между устройствами. |
| **Зачем существует?** | Wi-Fi существует, потому что радиоволна — это носитель, не требующий провода. Применив радиоволны из диапазона UHF/SHF к задаче передачи данных, мы получили способ связать устройства без кабеля. |
| **Как работает?** | Роутер берёт цифровые данные, модулирует ими радиосигнал на частоте 2,4/5/6 ГГц, излучает через антенну. Приёмник в устройстве ловит радиоволну, демодулирует и получает обратно данные. |
| **Почему именно так?** | Потому что волны длиной 5–12 см (диапазон UHF/SHF) проникают сквозь стены (с потерями), имеют малую антенну и позволяют передавать большие объёмы данных. Это оптимальный компромисс между дальностью, скоростью и размером устройств. |
| **Где применяется?** | Дома, в офисах, кафе, аэропортах, на заводах — везде, где нужно подключить устройство к сети без провода. |

---

## Шаг 1. Постановка проблемы

В предыдущей теме мы узнали: радиоволна — это электромагнитная волна с частотой 3 Гц – 300 ГГц, способная распространяться без среды. Она может нести энергию и, что важнее, **информацию**. Но как именно «положить» данные на радиоволну?

Представьте, что у вас дома 5 устройств: телефон, ноутбук, умный телевизор, умная колонка и игровой компьютер. Каждое хочет получить доступ в интернет.

Если использовать провода — придётся тянуть отдельный Ethernet-кабель от роутера к каждому устройству. Это 5 кабелей по квартире, отверстия в стенах, телефон привязан к розетке, телевизор нельзя переставить. А если пришёл гость с телефоном? Тянуть шестой кабель?

**Проблема:** как передавать данные между устройствами без физических проводов, так чтобы каждое устройство могло свободно перемещаться и подключаться по необходимости?

**Решение:** взять радиоволны из темы 1 (конкретно — диапазон UHF/SHF, частоты 2,4–6 ГГц), научиться «накладывать» на них цифровые данные и использовать как среду передачи. Эта идея реализована в стандарте IEEE 802.11, который называется **Wi-Fi**.

## Шаг 2. Практический пример

Давайте сначала «прощупаем» Wi-Fi программно. Мы не можем в Colab подключиться к настоящему роутеру (нет физического Wi-Fi-адаптера), но можем убедиться, что наш ноутбук «знает» про сеть, и смоделировать сканирование сетей как это делает операционная система.

In [ ]:
# Импортируем стандартные модули Python для работы с сетью
import socket
import platform
import os

# Узнаём имя хоста и его IP-адрес — это базовая информация о сетевом окружении
hostname = socket.gethostname()
try:
    local_ip = socket.gethostbyname(hostname)
except Exception as e:
    local_ip = f'(не удалось определить: {e})'

print(f'Имя хоста:     {hostname}')
print(f'Локальный IP:  {local_ip}')
print(f'ОС:            {platform.system()} {platform.release()}')
print()
print('Этот хост находится в некоторой IP-сети. В реальном окружении IP-адрес')
print('выдаёт Wi-Fi-роутер через DHCP — об этом мы поговорим в теме 5.')

**Разбор результата.** Если вы запустите этот код в Google Colab, вы увидите имя виртуальной машины (что-то вроде `abcdef123456`) и её внутренний IP. В Colab нет Wi-Fi-адаптера, но IP-адрес всё равно есть — он присвоен виртуальной сетевой картой. У вас дома ваш телефон получает аналогичный IP-адрес, но уже от Wi-Fi-роутера.

## Шаг 3. Интуитивное объяснение

Wi-Fi — это просто «невидимый провод». Когда вы подключаете телефон к роутеру по Wi-Fi, происходит то же самое, что и при подключении кабелем: данные идут от телефона к роутеру и обратно. Разница только в среде — вместо меди используется воздух.

Телефон и роутер «разговаривают» между собой радиоволнами. Каждое устройство имеет маленькую антенну, которая и «говорит», и «слушает». Эти радиоволны распространяются во все стороны, как круги по воде, и любое устройство в радиусе действия может их «услышать».

## Шаг 4. Жизненная аналогия

Представьте комнату, в которой 10 человек. Каждый хочет поговорить с каждым. Если все начнут говорить одновременно — наступит хаос, никто никого не услышит.

Wi-Fi решает эту проблему так же, как люди в комнате: каждое устройство «говорит» по очереди, короткими фразами, и обязательно представляется («Привет, это телефон Анны, я хочу отправить пакет»). Роутер — это как модератор в дискуссии, который раздаёт право голоса.

Если в комнате становится шумно (помехи от соседей), люди начинают говорить громче или медленнее. Wi-Fi делает то же самое: меняет скорость передачи и модуляцию, чтобы пробиться через помехи.

## Шаг 5. Техническое объяснение

Wi-Fi — это набор стандартов семейства **IEEE 802.11**. Они определяют, как именно устройства должны обмениваться радиосигналами, чтобы понимать друг друга.

Ключевые параметры сигнала:

- **Частота (frequency):** количество колебаний радиоволны в секунду. Wi-Fi работает на частотах 2,4 ГГц, 5 ГГц и (с 2020 года) 6 ГГц.
- **Длина волны (wavelength):** расстояние между двумя соседними гребнями волны. Связана с частотой формулой λ = c / f, где c = 3·10⁸ м/с — скорость света.
- **Модуляция (modulation):** способ «закодировать» цифровые нули и единицы в радиоволну.
- **Канал (channel):** узкая полоса частот внутри диапазона, на которой работает конкретная сеть.

Радиоволна — это электромагнитная волна. Чем выше частота, тем короче длина волны и тем меньше она огибает препятствия. Поэтому Wi-Fi на 5 ГГц даёт более высокую скорость, но хуже проходит сквозь стены, чем Wi-Fi на 2,4 ГГц.

## Шаг 6. Внутреннее устройство

Wi-Fi-система состоит из следующих компонентов:

| Компонент | Роль | Где находится |
|---|---|---|
| **Точка доступа (Access Point, AP)** | Эфирный «модератор», раздаёт Wi-Fi | Внутри роутера |
| **Клиент (Station, STA)** | Устройство, подключённое к AP | Телефон, ноутбук, IoT |
| **Антенна** | Преобразует электрический сигнал в радиоволну и обратно | Внутри всех устройств |
| **Радиомодуль (RF chip)** | Генерирует и принимает радиосигнал заданной частоты | Внутри устройств |
| **MAC-слой** | Управляет доступом к эфиру: кому говорить, когда говорить | Программно в драйвере |
| **PHY-слой** | Преобразует биты в радиосигнал через модуляцию | Программно в чипе |

Когда ваш телефон отправляет пакет в интернет, путь выглядит так:

1. Приложение формирует данные (например, HTTP-запрос).
2. ОС упаковывает их в IP-пакет.
3. Wi-Fi-драйвер оборачивает IP-пакет в Wi-Fi-кадр (802.11 frame).
4. Радиомодуль модулирует кадр в радиоволну.
5. Антенна излучает волну в эфир.
6. Антенна роутера принимает волну.
7. Радиомодуль роутера демодулирует её обратно в кадр.
8. Роутер извлекает IP-пакет и отправляет по кабелю в интернет.

И то же самое в обратную сторону для ответа.

## Шаг 7. Визуальная схема

Схема показывает путь пакета от приложения на телефоне до интернета через Wi-Fi.

```mermaid
flowchart LR
    A[Приложение<br/>на телефоне] --> B[ОС: IP-пакет]
    B --> C[Wi-Fi драйвер<br/>802.11 frame]
    C --> D[Радиомодуль<br/>модуляция]
    D --> E((Антенна<br/>телефона))
    E -.радиоволна.-> F((Антенна<br/>роутера))
    F --> G[Радиомодуль<br/>демодуляция]
    G --> H[Роутер: IP-пакет]
    H --> I[Кабель<br/>в Интернет]
    I --> J((Интернет))
```

Пунктирная линия — единственный «беспроводной» участок. Всё остальное — это обработка данных внутри устройств.

## Шаг 8. Рабочий пример

Посчитаем физические параметры радиоволны Wi-Fi на разных частотах. Этот пример работает в любом Python-окружении, включая Colab.

In [ ]:
# Расчёт длины волны Wi-Fi на разных частотах
# Формула: λ = c / f, где c — скорость света, f — частота

C = 3e8  # скорость света, м/с

freqs_ghz = {
    '2.4 ГГц (Wi-Fi 4)': 2.4e9,
    '5 ГГц (Wi-Fi 5/6)': 5.0e9,
    '6 ГГц (Wi-Fi 6E/7)': 6.0e9,
}

print(f"{'Диапазон':<25} {'Частота, ГГц':<15} {'Длина волны, см':<20}")
print('-' * 60)
for name, f in freqs_ghz.items():
    wavelength_m = C / f
    wavelength_cm = wavelength_m * 100
    print(f"{name:<25} {f/1e9:<15.1f} {wavelength_cm:<20.2f}")

print()
print('Вывод: чем выше частота, тем короче длина волны.')
print('Волна 12 см (2.4 ГГц) лучше огибает стены, чем волна 5 см (6 ГГц).')

**Разбор результата.** Мы получили, что Wi-Fi на 2,4 ГГц имеет длину волны ~12,5 см, а на 6 ГГц — ~5 см. Это объясняет физику проникновения сигнала сквозь стены: более длинная волна лучше огибает препятствия (дифракция), а короткая — отражается или поглощается. Поэтому 2,4 ГГц «бьёт» дальше, но медленнее, а 6 ГГц — быстрее, но в пределах одной комнаты.

## Шаг 9. Практический эксперимент

Рассчитаем затухание сигнала в зависимости от расстояния. Для расчёта используется формула **Free Space Path Loss (FSPL)** — потери в свободном пространстве:

**FSPL (dB) = 20·log₁₀(d) + 20·log₁₀(f) + 20·log₁₀(4π/c)**

где `d` — расстояние в метрах, `f` — частота в Гц, `c` — скорость света.

Меняйте переменную `distance` и наблюдайте, как растут потери.

In [ ]:
import math

C = 3e8  # скорость света

def fspl_db(distance_m: float, freq_hz: float) -> float:
    """Free Space Path Loss в децибелах."""
    return 20 * math.log10(distance_m) + 20 * math.log10(freq_hz) + 20 * math.log10(4 * math.pi / C)

# === ЭКСПЕРИМЕНТ: меняйте расстояние и частоту ===
distance = 10  # метры — попробуйте 1, 5, 10, 20, 50
frequency_hz = 2.4e9  # попробуйте 5e9, 6e9

loss = fspl_db(distance, frequency_hz)
print(f'Расстояние:    {distance} м')
print(f'Частота:       {frequency_hz/1e9} ГГц')
print(f'Потери FSPL:   {loss:.2f} дБ')
print()

# Сравним потери на разных расстояниях
print(f"{'Расстояние, м':<18}{'2.4 ГГц, дБ':<15}{'5 ГГц, дБ':<15}{'6 ГГц, дБ':<15}")
print('-' * 63)
for d in [1, 5, 10, 20, 50]:
    l24 = fspl_db(d, 2.4e9)
    l5 = fspl_db(d, 5e9)
    l6 = fspl_db(d, 6e9)
    print(f'{d:<18}{l24:<15.2f}{l5:<15.2f}{l6:<15.2f}')

**Что наблюдаем.** Каждое удвоение расстояния добавляет ~6 дБ потерь. Каждое удвоение частоты тоже добавляет ~6 дБ. Это значит: 6 ГГц на 10 м теряет примерно на 8 дБ больше, чем 2,4 ГГц на той же дистанции. Именно поэтому 5/6 ГГц-сети «бьют» короче, но дают выше скорость (за счёт более широкой полосы).

## 🧠 Проверка понимания

**1. (Объяснение)** Почему Wi-Fi на 2,4 ГГц проникает сквозь стены лучше, чем на 5 ГГц?

**2. (Прогноз)** Если увеличить расстояние от роутера с 5 м до 20 м (в 4 раза), насколько примерно вырастут потери сигнала в дБ?

**3. (Объяснение)** Зачем Wi-Fi-кадр оборачивает IP-пакет, а не передаёт IP-пакет напрямую в эфир?

---

## ⚠️ Частые ошибки

### Ошибка 1

**Неправильное рассуждение:** «Wi-Fi — это и есть интернет».

**Причина ошибки:** Пользователь путает локальную беспроводную сеть и глобальную сеть интернет.

**Правильное объяснение:** Wi-Fi — это только «последняя миля» между устройством и роутером. Роутер далее использует кабель (Ethernet/оптику) для доступа в интернет. Без провайдера Wi-Fi-сеть работает, но интернета не даёт.

### Ошибка 2

**Неправильное рассуждение:** «Чем выше частота, тем дальше бьёт сигнал».

**Причина ошибки:** Логика «выше = сильнее» не работает для радиоволн.

**Правильное объяснение:** Наоборот: чем выше частота, тем короче длина волны и тем сильнее она поглощается препятствиями. 2,4 ГГц бьёт дальше, но медленнее; 5/6 ГГц — быстрее, но короче.

### Ошибка 3

**Неправильное рассуждение:** «Wi-Fi и Bluetooth — это одно и то же».

**Причина ошибки:** Обе технологии беспроводные и работают в диапазоне 2,4 ГГц.

**Правильное объяснение:** Это разные стандарты с разными целями: Wi-Fi (802.11) — высокоскоростная локальная сеть; Bluetooth (802.15.1) — медленная связь между устройствами на коротком расстоянии (наушники, клавиатуры).

---

## 🎯 Краткий вывод

Wi-Fi — это набор стандартов беспроводной передачи данных через радиоволны на частотах 2,4 / 5 / 6 ГГц. Радиоволна имеет длину от 5 до 12 см, что определяет её способность проникать сквозь стены. Wi-Fi-система состоит из точки доступа, клиентов, антенн, радиомодулей и программных слоёв MAC/PHY. Потери сигнала в свободном пространстве растут логарифмически с расстоянием.

> 😄 Wi-Fi — это как крик в пустой комнате: чем дальше стоишь, тем хуже слышишь, а на 6 ГГц тебя ещё и стена хорошо заглушает.

---


## 🔗 Связь между темами

### Какие знания были получены

- Понятие радиоволны и её основные параметры (частота, длина волны)
- Структуру Wi-Fi-системы: AP, STA, антенны, радиомодуль, MAC и PHY
- Формулу FSPL и физику затухания сигнала

### Какие знания понадобятся далее

- Понимание каналов и ширины полосы
- Знание стандартов 802.11a/b/g/n/ac/ax/be
- Различие между диапазонами 2,4 / 5 / 6 ГГц

**Переход к следующей теме:** В теме 3 мы рассмотрим, как именно радиоволны Wi-Fi разделены на каналы, почему 2,4 ГГц «только 3 непересекающихся канала», и как эволюционировали стандарты Wi-Fi.

---

## ✅ Мини-проверка

**Вопрос 1** (предсказать результат):

Если вы уйдёте от роутера на 20 м вместо 10 м, насколько (примерно) вырастут потери сигнала по формуле FSPL?

**Вопрос 2** (объяснить причину):

Почему Wi-Fi на частоте 6 ГГц не может «пробить» бетонную стену так же хорошо, как 2,4 ГГц?

**Вопрос 3** (выбрать правильный вариант):

Что из перечисленного НЕ является компонентом Wi-Fi-системы? (a) Access Point (b) DHCP-сервер (c) Антенна (d) Радиомодуль

**Вопрос 4** (найти ошибку):

В коде `wavelength = 3e8 / 2.4` нашли длину волны для Wi-Fi 2,4 ГГц. Что не так?

---

## 🔬 Мини-лабораторная

**Цель:** Научиться рассчитывать длину волны для произвольной частоты Wi-Fi и сравнивать с длиной волны других радиосервисов.

### Пошаговая инструкция

1. Откройте новую ячейку в ноутбуке.
2. Используйте формулу λ = c / f, где c = 3·10⁸ м/с.
3. Рассчитайте длину волны для частот: 2,4 ГГц, 5 ГГц, 6 ГГц, 433 МГц (LPWAN), 13,56 МГц (NFC).
4. Выведите результаты в таблицу через pandas.
5. Постройте столбчатую диаграмму длин волн через matplotlib.

**Ожидаемый результат:** 5 значений длин волн в метрах (или см), таблица и график, где видно: чем ниже частота, тем длиннее волна.

**Объяснение результата:** Эта лабораторная закрепляет интуицию: частота и длина волны обратно пропорциональны. Wi-Fi 2,4 ГГц имеет волну ~12 см, NFC — ~22 м (намного длиннее, поэтому и действует на пару сантиметров).

---

## 🧪 Эксперимент

**Задание:** Измените расстояние в эксперименте из Шага 9 и проверьте, выполняется ли правило «удвоение расстояния = +6 дБ».

Изменяемые параметры:

- distance = 1, 2, 4, 8, 16, 32 м
- frequency_hz: попробуйте 2.4e9 и 5.8e9

**Что наблюдаем:** Разница потерь между соседними удвоениями должна быть около 6,02 дБ. Если это так — вы убедились в законе FSPL на практике.

---

## 📝 Тестовые задания с вариантами ответов

> Выберите один правильный вариант в каждом задании. Ответы — в конце блока.

### Задание 1

Какое утверждение о частоте Wi-Fi верное?

- [ ] **a)** Чем выше частота, тем дальше бьёт сигнал
- [ ] **b)** Чем выше частота, тем короче длина волны
- [ ] **c)** Wi-Fi работает только на частоте 2,4 ГГц
- [ ] **d)** Частота Wi-Fi измеряется в ваттах

### Задание 2

Что такое FSPL?

- [ ] **a)** Fast Signal Protocol Layer — протокол MAC
- [ ] **b)** Free Space Path Loss — потери сигнала в свободном пространстве
- [ ] **c)** Frequency Spectrum Power Level — уровень мощности частоты
- [ ] **d)** Frame Sequence Packet Length — длина кадра

### Задание 3

Какой компонент НЕ входит в Wi-Fi-систему?

- [ ] **a)** Access Point
- [ ] **b)** Station (STA)
- [ ] **c)** Антенна
- [ ] **d)** Жёсткий диск (HDD)

### Задание 4

Если расстояние до роутера увеличилось в 4 раза, потери сигнала вырастут примерно на:

- [ ] **a)** 3 дБ
- [ ] **b)** 6 дБ
- [ ] **c)** 12 дБ
- [ ] **d)** 24 дБ

### Задание 5

Длина волны Wi-Fi 5 ГГц равна примерно:

- [ ] **a)** 12 см
- [ ] **b)** 6 см
- [ ] **c)** 5 см
- [ ] **d)** 25 см

<details><summary><b>🔑 Ответы и пояснения (нажмите, чтобы развернуть)</b></summary>

**Задание 1:** правильный ответ — **b)**. Длина волны λ = c / f, поэтому рост частоты ведёт к уменьшению длины волны. Бóльшая частота = короче волна = хуже проникновение через стены, но выше потенциальная скорость.

**Задание 2:** правильный ответ — **b)**. FSPL описывает, насколько слабее сигнал становится при распространении в свободном пространстве. Растёт логарифмически с расстоянием: +6 дБ на каждое удвоение дистанции.

**Задание 3:** правильный ответ — **d)**. Access Point, Station и антенна — обязательные элементы Wi-Fi. Жёсткий диск не имеет отношения к беспроводной передаче данных.

**Задание 4:** правильный ответ — **c)**. Удвоение расстояния даёт +6 дБ. Четырёхкратное увеличение — это два удвоения, то есть +12 дБ.

**Задание 5:** правильный ответ — **b)**. λ = 3·10⁸ / 5·10⁹ = 0,06 м = 6 см. Это короче, чем у 2,4 ГГц (~12,5 см), поэтому 5 ГГц хуже проходит через стены.

</details>

---

## ✅ Чек-лист темы

- Я понимаю, что Wi-Fi — это радиосвязь, а не «магический интернет».
- Я понимаю, что частота и длина волны обратно пропорциональны.
- Я умею рассчитывать длину волны по формуле λ = c / f.
- Я умею рассчитывать потери сигнала FSPL для заданного расстояния.
- Я могу объяснить, почему 2,4 ГГц «пробивает» стены лучше, чем 5 ГГц.
- Я могу объяснить путь пакета от приложения на телефоне до интернета.

---

# 📘 Тема 3. Стандарты Wi-Fi и частоты: 2.4 / 5 / 6 ГГц

## Пять вопросов темы

| Вопрос | Ответ |
|---|---|
| **Что это?** | Стандарты Wi-Fi — это спецификации IEEE 802.11 (a, b, g, n, ac, ax, be), определяющие скорости, частоты и технологии модуляции. |
| **Зачем существует?** | Стандарты нужны, чтобы устройства разных производителей понимали друг друга и чтобы новые технологии могли развиваться, не ломая совместимость со старым оборудованием. |
| **Как работает?** | Каждый стандарт задаёт: рабочую частоту, ширину канала, схему модуляции, число антенн (MIMO) и методы множественного доступа. |
| **Почему именно так?** | Потому что радиоспектр — ограниченный ресурс, и его нужно делить между устройствами и приложениями. Стандарты описывают, как именно делить. |
| **Где применяется?** | В каждом Wi-Fi-устройстве: роутере, телефоне, ноутбуке, телевизоре, IoT-датчике. |

---

## Шаг 1. Постановка проблемы

Представьте: у вас дома телефон 2024 года, ноутбук 2015-го и старый принтер 2010-го. Все три должны подключиться к одному роутеру. Но они сделаны в разное время и поддерживают разные технологии. Как роутеру «договориться» с каждым из них?

Если бы стандартов не было — каждый производитель делал бы Wi-Fi по-своему. Wi-Fi-роутер Asus не работал бы с телефоном Samsung, а телефон Samsung — с принтером HP. Ситуация как в басне Крылова: «Лебедь, рак и щука».

**Проблема:** как сделать так, чтобы любое Wi-Fi-устройство понимало любое другое, даже если они выпущены в разное десятилетие разными производителями?

**Решение:** создать международные стандарты IEEE 802.11, которые описывают все детали — от частоты до формата кадра. Производители обязаны их соблюдать.

## Шаг 2. Практический пример

Сымитируем сканирование эфира, как это делает ОС при поиске сетей. В Colab нет реального Wi-Fi-адаптера, поэтому мы сгенерируем mock-данные, похожие на те, что вернула бы команда `iwlist scan` в Linux или `netsh wlan show networks` в Windows.

In [ ]:
import random
import pandas as pd

random.seed(42)

# Имитируем список Wi-Fi сетей, которые «видит» адаптер
ssids = ['Home_5G', 'Home_2G', 'Neighbor_1', 'Neighbor_2', 'Cafe_Guest', 'IoT_Sensor', 'Printer_HP']
standards = ['802.11ax', '802.11ac', '802.11n', '802.11g']
channels_24 = list(range(1, 14))
channels_5 = [36, 40, 44, 48, 149, 153, 157, 161]

networks = []
for ssid in ssids:
    band = random.choice(['2.4', '5'])
    channel = random.choice(channels_24 if band == '2.4' else channels_5)
    networks.append({
        'SSID': ssid,
        'BSSID': f'{random.randint(0,255):02X}:{random.randint(0,255):02X}:{random.randint(0,255):02X}:'
                 f'{random.randint(0,255):02X}:{random.randint(0,255):02X}:{random.randint(0,255):02X}',
        'Стандарт': random.choice(standards),
        'Диапазон': f'{band} ГГц',
        'Канал': channel,
        'RSSI, дБм': random.randint(-90, -40),
        'Шифрование': random.choice(['WPA3', 'WPA2', 'WPA2/WPA3'])
    })

df = pd.DataFrame(networks)
df

**Разбор результата.** В таблице видны ключевые поля, которые ОС получает от каждого «увиденного» Wi-Fi-роутера: SSID (имя сети), BSSID (MAC-адрес роутера), стандарт, диапазон, номер канала, уровень сигнала RSSI и тип шифрования. По этой информации ОС решает, к какой сети подключиться.

## Шаг 3. Интуитивное объяснение

Wi-Fi-стандарты — это как поколения мобильной связи: 3G, 4G, 5G. Каждое новое поколение быстрее и умнее предыдущего, но обратно совместимо: телефон 5G может работать в сети 4G, если 5G нет.

В Wi-Fi поколения называются сложно: 802.11b, 802.11g, 802.11n, 802.11ac, 802.11ax, 802.11be. Чтобы не путаться, Wi-Fi Alliance ввёл простые имена:

- **Wi-Fi 4** = 802.11n (2009)
- **Wi-Fi 5** = 802.11ac (2013)
- **Wi-Fi 6 / 6E** = 802.11ax (2019 / 2020)
- **Wi-Fi 7** = 802.11be (2024)

## Шаг 4. Жизненная аналогия

Представьте сеть дорог. Старые дороги (802.11b) — это однополосная грунтовка: медленно, но доедешь куда угодно. Wi-Fi 5 — это 4-полосная трасса, по которой быстро, но въезд только с машин определённого типа. Wi-Fi 6 — это современная автомагистраль с 8 полосами, разделёнными эстакадами, где грузовые и легковые едут одновременно и не мешают друг другу.

Wi-Fi 7 — это «умная» магистраль, где несколько машин могут ехать бок о бок в одной полосе (Multi-Link Operation), мгновенно переключаясь между полосами без остановки.

## Шаг 5. Техническое объяснение

Сравним характеристики стандартов:

| Стандарт | Имя | Год | Частоты | Макс. скорость | Ширина канала | MIMO |
|---|---|---|---|---|---|---|
| 802.11b | — | 1999 | 2,4 ГГц | 11 Мбит/с | 22 МГц | нет |
| 802.11g | — | 2003 | 2,4 ГГц | 54 Мбит/с | 20 МГц | нет |
| 802.11n | Wi-Fi 4 | 2009 | 2,4 / 5 ГГц | 600 Мбит/с | 20/40 МГц | 4×4 |
| 802.11ac | Wi-Fi 5 | 2013 | 5 ГГц | 3,5 Гбит/с | 20/40/80/160 МГц | 8×8 |
| 802.11ax | Wi-Fi 6/6E | 2019/2020 | 2,4 / 5 / 6 ГГц | 9,6 Гбит/с | до 160 МГц | 8×8 + OFDMA |
| 802.11be | Wi-Fi 7 | 2024 | 2,4 / 5 / 6 ГГц | 46 Гбит/с | до 320 МГц | 16×16 + MLO |

Ключевые технологии, которые появились в разных поколениях:

- **MIMO** (Multiple Input Multiple Output) — несколько антенн передают и принимают одновременно. Появилось в Wi-Fi 4.
- **MU-MIMO** (Multi-User MIMO) — несколько антенн обслуживают несколько устройств одновременно. Wi-Fi 5 (downlink) и Wi-Fi 6 (uplink + downlink).
- **OFDMA** — делит канал на подканалы, чтобы обслуживать много мелких устройств разом. Wi-Fi 6.
- **MLO** (Multi-Link Operation) — устройство одновременно использует 2,4 + 5 + 6 ГГц. Wi-Fi 7.

## Шаг 6. Внутреннее устройство

Радиоспектр Wi-Fi разделён на **диапазоны (bands)** и **каналы (channels)**:

| Диапазон | Полоса частот | Кол-во каналов | Ширина канала |
|---|---|---|---|
| 2,4 ГГц | 2400–2483,5 МГц | 14 (13 в РФ/Европе) | 20 МГц (22 МГц реально) |
| 5 ГГц | 5150–5895 МГц | ~25 (зависит от страны) | 20/40/80/160 МГц |
| 6 ГГц | 5925–7125 МГц | до 59 (зависит от страны) | 20/40/80/160/320 МГц |

**Критически важный факт про 2,4 ГГц:** каналы перекрываются! Если два роутера стоят на каналах 1 и 2, их сигналы частично «наезжают» друг на друга. Только каналы 1, 6 и 11 не пересекаются между собой. Именно их и нужно использовать.

В диапазонах 5 и 6 ГГц каналы не перекрываются (между ними есть зазоры), поэтому там выбор канала проще.

## Шаг 7. Визуальная схема

Эволюция стандартов Wi-Fi:

```mermaid
timeline
    title Эволюция стандартов Wi-Fi
    1999 : 802.11b (Wi-Fi 1) — 11 Мбит/с, 2.4 ГГц
    2003 : 802.11g (Wi-Fi 3) — 54 Мбит/с, 2.4 ГГц
    2009 : 802.11n (Wi-Fi 4) — 600 Мбит/с, 2.4+5 ГГц, MIMO
    2013 : 802.11ac (Wi-Fi 5) — 3.5 Гбит/с, 5 ГГц, MU-MIMO
    2019 : 802.11ax (Wi-Fi 6) — 9.6 Гбит/с, OFDMA
    2020 : 802.11ax-6E (Wi-Fi 6E) — добавлен диапазон 6 ГГц
    2024 : 802.11be (Wi-Fi 7) — 46 Гбит/с, MLO
```

Перекрытие каналов в 2,4 ГГц:

```mermaid
flowchart LR
    subgraph C1[Канал 1]
        A1[2412 МГц]
    end
    subgraph C2[Канал 2]
        A2[2417 МГц — перекрытие!]
    end
    subgraph C6[Канал 6]
        A6[2437 МГц — чисто]
    end
    subgraph C11[Канал 11]
        A11[2462 МГц — чисто]
    end
    C1 -.перекрытие.-> C2
    C1 -.нет перекрытия.-> C6
    C6 -.нет перекрытия.-> C11
```

## Шаг 8. Рабочий пример

Визуализируем перекрытие каналов 2,4 ГГц. Каждый канал занимает ~22 МГц, а расстояние между центрами — 5 МГц. Это значит, что соседние каналы неизбежно перекрываются.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Поиск шрифта с поддержкой кириллицы по нескольким известным путям
_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf',
    '/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]
for _p in _FONT_CANDIDATES:
    if os.path.exists(_p):
        try:
            fm.fontManager.addfont(_p)
        except Exception:
            pass
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Liberation Sans', 'FreeSans']
plt.rcParams['axes.unicode_minus'] = False

def channel_spectrum(channel: int, width_mhz: float = 22.0) -> tuple:
    """Возвращает (частоты, мощности) для типичного Wi-Fi канала."""
    center = 2407 + 5 * channel  # 2412 для канала 1, 2437 для канала 6
    f = np.linspace(center - width_mhz, center + width_mhz, 200)
    power = np.exp(-((f - center) / (width_mhz / 2.355)) ** 2)
    return f, power

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)

# Покажем каналы 1, 2, 6 и 11
for ch, color, label in [
    (1, '#2563EB', 'Канал 1'),
    (2, '#DC2626', 'Канал 2 (перекрытие с 1!)'),
    (6, '#16A34A', 'Канал 6 (чистый)'),
    (11, '#9333EA', 'Канал 11 (чистый)'),
]:
    f, p = channel_spectrum(ch)
    ax.plot(f, p, label=label, color=color, linewidth=2)
    ax.fill_between(f, p, alpha=0.15, color=color)

ax.set_xlabel('Частота, МГц', fontsize=11)
ax.set_ylabel('Относительная мощность', fontsize=11)
ax.set_title('Перекрытие каналов Wi-Fi 2,4 ГГц', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlim(2400, 2480)
ax.grid(True, alpha=0.3)
plt.show()

**Разбор результата.** На графике видно: «колокол» канала 2 сильно перекрывается с каналом 1. Если ваш роутер стоит на канале 1, а соседский — на канале 2, вы будете мешать друг другу. А вот каналы 1, 6 и 11 разделены достаточно, чтобы не пересекаться. Это и есть правило трёх непересекающихся каналов.

## Шаг 9. Практический эксперимент

Посчитаем, сколько сетей в среднем «давят» друг на друга в зависимости от выбранного канала. Меняйте количество соседей и наблюдайте.

In [ ]:
import random
from collections import Counter

random.seed(7)

# === ЭКСПЕРИМЕНТ: меняйте число соседей ===
num_neighbors = 10  # попробуйте 5, 10, 20, 30

# Соседи случайно выбирают каналы 1..13
neighbor_channels = [random.randint(1, 13) for _ in range(num_neighbors)]

def interference(target: int, channels: list[int]) -> int:
    """Считаем, сколько соседей перекрываются с целевым каналом."""
    return sum(1 for c in channels if abs(c - target) < 5)

print(f'Соседей: {num_neighbors}, их каналы: {sorted(neighbor_channels)}')
print()
print(f"{'Канал':<10}{'Перекрытие с соседями':<25}")
print('-' * 35)
for ch in [1, 6, 11]:
    n = interference(ch, neighbor_channels)
    bar = '█' * n
    print(f'{ch:<10}{n:<5} {bar}')

print()
print('Вывод: каналы 1, 6 и 11 — лучшие. Если у вас много соседей, выбирайте наименее загруженный.')

## 🧠 Проверка понимания

**1. (Объяснение)** Почему в 2,4 ГГц «всего 3 непересекающихся канала», а в 5 ГГц — больше 20?

**2. (Прогноз)** Если сосед поставил роутер на канал 4, какие ваши каналы получат помехи?

**3. (Объяснение)** Зачем в Wi-Fi 6 добавили OFDMA, если уже было MU-MIMO в Wi-Fi 5?

---

## ⚠️ Частые ошибки

### Ошибка 1

**Неправильное рассуждение:** «Wi-Fi 6 — это и есть 6 ГГц».

**Причина ошибки:** Путаница в номерах: Wi-Fi 6 и диапазон 6 ГГц — не одно и то же.

**Правильное объяснение:** Wi-Fi 6 (802.11ax) работает на 2,4 и 5 ГГц. Wi-Fi 6E — это расширение, добавившее диапазон 6 ГГц. Wi-Fi 7 работает во всех трёх диапазонах.

### Ошибка 2

**Неправильное рассуждение:** «Чем выше номер канала, тем выше скорость».

**Причина ошибки:** Логика «больше = лучше» не работает для каналов.

**Правильное объяснение:** Номер канала — это просто номер центральной частоты. Скорость зависит от ширины канала (20/40/80/160 МГц) и стандарта, а не от номера.

### Ошибка 3

**Неправильное рассуждение:** «5 ГГц всегда быстрее, чем 2,4 ГГц».

**Причина ошибки:** Сравнивают только теоретическую скорость.

**Правильное объяснение:** 5 ГГц быстрее при сильном сигнале и малом расстоянии. Но если расстояние большое или есть стены — 2,4 ГГц может оказаться быстрее за счёт лучшего проникновения.

---

## 🎯 Краткий вывод

Стандарты Wi-Fi эволюционируют: каждое новое поколение (Wi-Fi 4 → 5 → 6 → 7) добавляет скорости и эффективности через MIMO, MU-MIMO, OFDMA и MLO. В диапазоне 2,4 ГГц только 3 непересекающихся канала (1, 6, 11), в 5 и 6 ГГц каналов больше и они не перекрываются. Выбор правильного канала — основа стабильной работы сети.

> 😄 В 2,4 ГГц 13 каналов, но мирятся друг с другом только 3 из них — как три родственника в одной кухне: если пригласить четвёртого, начнётся свара.

---


## 🔗 Связь между темами

### Какие знания были получены

- Поколения стандартов Wi-Fi: 4, 5, 6, 6E, 7 и их характеристики
- Диапазоны 2,4 / 5 / 6 ГГц и их особенности
- Концепцию каналов и перекрытия в 2,4 ГГц

### Какие знания понадобятся далее

- Понимание, как защищать Wi-Fi-сеть от подслушивания
- Знание протоколов шифрования WPA2 и WPA3
- Понимание процесса аутентификации (4-way handshake)

**Переход к следующей теме:** В теме 4 мы разберём, как именно Wi-Fi защищает передаваемые данные от перехвата, почему WEP уже не используется, чем WPA3 лучше WPA2 и как работает 4-way handshake.

---

## ✅ Мини-проверка

**Вопрос 1** (объяснить причину):

Почему каналы 1, 6 и 11 в 2,4 ГГц считаются «непересекающимися»?

**Вопрос 2** (предсказать результат):

Если сосед ставит сеть на канал 4, на каких ваших каналах будет помеха?

**Вопрос 3** (выбрать правильный вариант):

Какой стандарт работает в диапазоне 6 ГГц? (a) Wi-Fi 4 (b) Wi-Fi 5 (c) Wi-Fi 6E (d) Wi-Fi 3

**Вопрос 4** (объяснить последовательность действий):

Как вы будете выбирать канал для нового роутера в многоквартирном доме?

---

## 🔬 Мини-лабораторная

**Цель:** Научиться определять оптимальный канал Wi-Fi по результатам сканирования эфира.

### Пошаговая инструкция

1. Сгенерируйте mock-данные 20 Wi-Fi сетей с случайными каналами (1..13).
2. Для каждого канала от 1 до 13 посчитайте суммарную «помеху» — сколько соседних каналов перекрывается.
3. Постройте столбчатую диаграмму: по оси X — номер канала, по оси Y — уровень помех.
4. Найдите канал с минимальной помехой.
5. Выведите рекомендацию: «Лучший канал: X».

**Ожидаемый результат:** График с 13 столбцами, минимум — на каналах 1, 6 или 11.

**Объяснение результата:** Эта лабораторная учит анализировать эфир перед выбором канала. В реальной жизни это делают приложения вроде WiFi Analyzer (Android) или `airport` (macOS).

---

## 🧪 Эксперимент

**Задание:** В коде из Шага 9 меняйте `num_neighbors` от 5 до 30 и наблюдайте, как меняется «лучший канал».

Изменяемые параметры:

- num_neighbors = 5, 10, 20, 30
- Подумайте: что произойдёт, если все соседи выберут канал 6?

**Что наблюдаем:** Чем больше соседей, тем вероятнее, что каналы 1, 6 и 11 тоже станут загруженными. В сверхплотной застройке 2,4 ГГц практически неработоспособен.

---

## 📝 Тестовые задания с вариантами ответов

> Выберите один правильный вариант в каждом задании. Ответы — в конце блока.

### Задание 1

Сколько непересекающихся каналов в диапазоне 2,4 ГГц (в РФ)?

- [ ] **a)** 1
- [ ] **b)** 3
- [ ] **c)** 7
- [ ] **d)** 13

### Задание 2

Какая технология появилась в Wi-Fi 6?

- [ ] **a)** MIMO
- [ ] **b)** OFDMA
- [ ] **c)** MLO
- [ ] **d)** Beamforming

### Задание 3

Wi-Fi 6E — это:

- [ ] **a)** Расширение Wi-Fi 6 на диапазон 6 ГГц
- [ ] **b)** Энергосберегающая версия Wi-Fi 6
- [ ] **c)** Старый стандарт 802.11e
- [ ] **d)** Enterprise-версия Wi-Fi 6 для офисов

### Задание 4

Если ваш роутер на канале 6, а соседский — на канале 8, то:

- [ ] **a)** Помех не будет
- [ ] **b)** Будут помехи — каналы перекрываются
- [ ] **c)** Скорость увеличится
- [ ] **d)** Роутер автоматически переключится на канал 11

### Задание 5

Максимальная теоретическая скорость Wi-Fi 7:

- [ ] **a)** 600 Мбит/с
- [ ] **b)** 3,5 Гбит/с
- [ ] **c)** 9,6 Гбит/с
- [ ] **d)** 46 Гбит/с

<details><summary><b>🔑 Ответы и пояснения (нажмите, чтобы развернуть)</b></summary>

**Задание 1:** правильный ответ — **b)**. Каналы 1, 6 и 11 не перекрываются между собой. Это единственный безопасный набор из 3 каналов в 2,4 ГГц.

**Задание 2:** правильный ответ — **b)**. OFDMA (Orthogonal Frequency-Division Multiple Access) — ключевая технология Wi-Fi 6, позволяющая делить канал на подканалы для обслуживания многих мелких устройств разом. MIMO было в Wi-Fi 4, MLO — в Wi-Fi 7.

**Задание 3:** правильный ответ — **a)**. Wi-Fi 6E = Wi-Fi 6 + новый диапазон 6 ГГц (5925–7125 МГц). Дал много новых непересекающихся каналов.

**Задание 4:** правильный ответ — **b)**. Канал 8 центрирован на 2447 МГц, канал 6 — на 2437 МГц. Разница 10 МГц, а ширина каждого канала ~22 МГц — перекрытие гарантировано.

**Задание 5:** правильный ответ — **d)**. Wi-Fi 7 (802.11be) заявляет до 46 Гбит/с за счёт ширины канала 320 МГц, 16×16 MIMO и Multi-Link Operation.

</details>

---

## ✅ Чек-лист темы

- Я понимаю разницу между Wi-Fi 4, 5, 6, 6E и 7.
- Я понимаю, почему в 2,4 ГГц только 3 непересекающихся канала.
- Я умею визуализировать перекрытие каналов через matplotlib.
- Я умею анализировать mock-данные сканирования Wi-Fi через pandas.
- Я могу объяснить, что такое OFDMA и зачем он нужен.
- Я могу объяснить, чем Wi-Fi 6E отличается от Wi-Fi 6.

---

# 📘 Тема 4. Безопасность Wi-Fi: WPA2, WPA3, шифрование

## Пять вопросов темы

| Вопрос | Ответ |
|---|---|
| **Что это?** | Безопасность Wi-Fi — это набор протоколов (WEP → WPA → WPA2 → WPA3), которые шифруют радиосигнал и проверяют личность подключающихся устройств. |
| **Зачем существует?** | Радиоволны распространяются во все стороны — любой в радиусе может «слушать» эфир. Без шифрования ваш банковский пароль перехватит сосед с ноутбуком и Wi-Fi-адаптером. |
| **Как работает?** | Устройство и роутер договариваются о ключе шифрования через 4-way handshake, после чего все кадры между ними шифруются симметричным алгоритмом (AES). |
| **Почему именно так?** | Потому что радиоканал — это «общая» среда, доступная всем. Нужна криптография, доступная только легитимным участникам. |
| **Где применяется?** | В каждой Wi-Fi-сети: домашней, корпоративной, общественной (гостевой). |

---

## Шаг 1. Постановка проблемы

Вы зашли в кафе, открыли ноутбук и увидели открытую сеть `Cafe_Guest` (без пароля). Подключились, зашли в банк. Удобно? Удобно.

Проблема в том, что в этом же кафе сидит человек с программой Wireshark и Wi-Fi-адаптером в режиме монитора. Он перехватывает все радиосигналы в эфире — включая ваши. Если сеть без шифрования, ваш HTTP-запрос к банку он видит буквально в открытом виде.

**Проблема:** как сделать так, чтобы перехваченный радиосигнал нельзя было прочитать без ключа, и при этом не усложнять подключение к сети для обычного пользователя?

**Решение:** использовать криптографические протоколы WPA2/WPA3, которые:
- проверяют, что вы знаете пароль (без передачи самого пароля);
- вырабатывают временный ключ шифрования для каждой сессии;
- шифруют каждый кадр так, что перехват бесполезен.

## Шаг 2. Практический пример

Шифрование — это не магия. Покажем на Python, как обычный текст превращается в шифротекст и обратно. Используем симметричное шифрование AES — тот же алгоритм, что в WPA2.

In [ ]:
# Демонстрация симметричного шифрования AES (как в WPA2)
# pip install cryptography  (если ещё не установлен)
import os
import hashlib
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

# В WPA2 ключ шифрования вырабатывается из пароля через PBKDF2
wifi_password = 'MySuperSecretWifi2024'
ssid = 'HomeWiFi'
# PMK (Pairwise Master Key) — 32 байта, вычисляется один раз
pmk = hashlib.pbkdf2_hmac('sha1', wifi_password.encode(), ssid.encode(), 4096, 32)
print(f'PMK (первые 16 байт): {pmk[:16].hex()}')
print(f'PMK полный:           {pmk.hex()}')
print()

# Симметричное шифрование AES-CCMP (как в WPA2)
def aes_ccmp_encrypt(key: bytes, plaintext: bytes, nonce: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key), modes.CTR(nonce), backend=default_backend())
    enc = cipher.encryptor()
    return enc.update(plaintext) + enc.finalize()

def aes_ccmp_decrypt(key: bytes, ciphertext: bytes, nonce: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key), modes.CTR(nonce), backend=default_backend())
    dec = cipher.decryptor()
    return dec.update(ciphertext) + dec.finalize()

# Сообщение, которое мы «отправляем по Wi-Fi»
message = b'Hello, bank! Transfer 1000 USD to account 1234567890.'
nonce = os.urandom(16)  # уникальный для каждого пакета

ciphertext = aes_ccmp_encrypt(pmk, message, nonce)
print(f'Открытый текст:  {message.decode()}')
print(f'Шифротекст (hex): {ciphertext.hex()[:80]}...')
print()

# Легитимный получатель с тем же ключом — расшифрует
decrypted = aes_ccmp_decrypt(pmk, ciphertext, nonce)
print(f'Расшифровано:    {decrypted.decode()}')
print()
print('Без знания PMK перехватчик увидит только белиберду.')

**Разбор результата.** Пароль `MySuperSecretWifi2024` прошёл через функцию PBKDF2 (4096 итераций SHA-1) и превратился в 32-байтовый ключ PMK. Этот ключ далее используется для AES-шифрования. Перехватчик видит только шифротекст — набор случайных байтов. Без знания PMK восстановить открытый текст невозможно.

## Шаг 3. Интуитивное объяснение

Wi-Fi-шифрование работает как сейф: ключ есть только у тех, кто знает пароль. Все остальные видят запертый сейф, но не его содержимое.

Процесс подключения к защищённой сети:
1. Вы вводите пароль.
2. Телефон и роутер устраивают «криптографическую беседу» (4-way handshake), не раскрывая пароль.
3. По итогам беседы оба вычисляют одинаковый временный ключ.
4. Дальше весь трафик шифруется этим ключом.

Даже если перехватчик записал всю беседу, без пароля он не сможет вычислить ключ. Это и есть основа безопасности WPA2/WPA3.

## Шаг 4. Жизненная аналогия

Представьте, что вы хотите отправить посылку с деньгами другу через курьера, которому не доверяете. У вас и у друга есть одинаковый замок и ключ.

Вы кладёте деньги в коробку, закрываете своим замком и отправляете. Курьер везёт закрытую коробку — он не может её открыть. Друг получает посылку, но его ключ не подходит к вашему замку.

Чтобы это работало, вы заранее договорились: вы закрываете коробку своим замком, друг добавляет свой замок и отправляет обратно. Вы снимаете свой замок, отправляете снова — друг снимает свой. Это принцип 4-way handshake, только с криптографией вместо физических замков.

## Шаг 5. Техническое объяснение

Эволюция протоколов безопасности Wi-Fi:

| Протокол | Год | Шифр | Аутентификация | Статус |
|---|---|---|---|---|
| **WEP** | 1997 | RC4 (40/104 бит) | Статический ключ | ❌ Взломан за минуты |
| **WPA** | 2003 | RC4 + TKIP | PSK или 802.1X | ❌ Уязвим |
| **WPA2** | 2004 | AES-CCMP | PSK или 802.1X | ⚠️ SAE в WPA3 решает проблемы |
| **WPA3** | 2018 | AES-CCMP + GCMP | SAE (Simultaneous Authentication of Equals) | ✅ Современный |

Ключевые термины:

- **PSK (Pre-Shared Key)** — режим с общим паролем (домашние сети).
- **802.1X / Enterprise** — режим с сервером аутентификации RADIUS (корпоративные сети).
- **PMK (Pairwise Master Key)** — главный ключ сессии, вычисляется из пароля.
- **PTK (Pairwise Transient Key)** — временный ключ шифрования, вырабатывается в 4-way handshake.
- **GTK (Group Temporal Key)** — ключ для широковещательного трафика (beacon, multicast).
- **4-way handshake** — процедура из 4 сообщений между клиентом и AP для выработки PTK.
- **SAE** — современная замена PSK в WPA3, устойчивая к офлайн-перебору паролей.

## Шаг 6. Внутреннее устройство

Как работает 4-way handshake в WPA2-PSK:

1. **PMK** вычисляется один раз: `PMK = PBKDF2-HMAC-SHA1(пароль, SSID, 4096 итераций, 32 байта)`.
2. Клиент и AP обмениваются случайными числами `ANonce` (от AP) и `SNonce` (от клиента).
3. Обе стороны вычисляют **PTK** по формуле: `PTK = PRF(PMK, ANonce, SNonce, MAC_AP, MAC_STA)`.
4. PTK состоит из 3 частей: `KCK` (проверка целостности), `KEK` (шифрование GTK), `TK` (шифрование данных).
5. AP отправляет GTK, зашифрованный KEK.
6. Клиент подтверждает установку ключей.

В WPA3 вместо PSK используется **SAE** (Dragonfly protocol) — даже при перехвате handshake нельзя атаковать пароль офлайн. Это делает WPA3 стойким к атакам перебором.

## Шаг 7. Визуальная схема

Диаграмма последовательности 4-way handshake в WPA2:

```mermaid
sequenceDiagram
    participant C as Клиент (телефон)
    participant A as Точка доступа (роутер)
    Note over C,A: Оба знают SSID и пароль → вычислили PMK
    A->>C: Сообщение 1: ANonce (случайное число AP)
    Note over C: Клиент генерирует SNonce, вычисляет PTK
    C->>A: Сообщение 2: SNonce + MIC (доказательство знания PTK)
    Note over A: AP вычисляет PTK, проверяет MIC
    A->>C: Сообщение 3: GTK (зашифрован KEK) + MIC
    Note over C: Клиент расшифровывает GTK, устанавливает ключи
    C->>A: Сообщение 4: Подтверждение (ACK)
    Note over C,A: Канал установлен, трафик шифруется TK
```

Схема поколений протоколов безопасности:

```mermaid
flowchart LR
    W[WEP<br/>1997<br/>RC4 40 бит] -->|взломан| WP[WPA<br/>2003<br/>RC4+TKIP]
    WP -->|уязвимости| W2[WPA2<br/>2004<br/>AES-CCMP]
    W2 -->|офлайн-перебор| W3[WPA3<br/>2018<br/>SAE+GCMP]
    style W fill:#fecaca
    style WP fill:#fed7aa
    style W2 fill:#fef3c7
    style W3 fill:#bbf7d0
```

## Шаг 8. Рабочий пример

Покажем, как длина пароля влияет на время взлома. WPA2 уязвим к офлайн-перебору: перехватив handshake, атакующий может пробовать пароли без обращения к роутеру. Скорость перебора на современной GPU-ферме — ~1 млн PMK в секунду.

In [ ]:
import math

# Скорость перебора WPA2-PMK на современной GPU-ферме
# Источник: benchmarks hashcat на RTX 4090 ~ 2.5M PMK/s на одну карту
PMK_PER_SECOND = 2_500_000

def crack_time(password_entropy_bits: float, rate: int = PMK_PER_SECOND) -> float:
    """Возвращает среднее время полного перебора, секунд."""
    keyspace = 2 ** password_entropy_bits
    return keyspace / (2 * rate)  # в среднем перебор останавливается на середине

def format_time(seconds: float) -> str:
    if seconds < 60: return f'{seconds:.1f} секунд'
    if seconds < 3600: return f'{seconds/60:.1f} минут'
    if seconds < 86400: return f'{seconds/3600:.1f} часов'
    if seconds < 86400*365: return f'{seconds/86400:.1f} дней'
    if seconds < 86400*365*1000: return f'{seconds/(86400*365):.1f} лет'
    if seconds < 86400*365*1e9: return f'{seconds/(86400*365*1e6):.1f} млн лет'
    return f'{seconds/(86400*365*1e9):.2e} млрд лет'

# Сравним пароли разной сложности
passwords = [
    ('12345678',          26.4),   # 8 цифр
    ('qwerty12',          41.4),   # 8 символов lowercase+digits
    ('MyWifi2024',        57.0),   # 10 символов, смешанный
    ('Tr0ub4dour&3',      71.0),   # 12 символов, сложный
    ('correct-horse-battery-staple', 100.0),  # 4 случайных слова
    ('aB3$xQ9!pL7#mN2&vR5*', 131.0),  # 20 символов, все классы
]

print(f"{'Пароль':<32}{'Энтропия, бит':<18}{'Время взлома'}")
print('-' * 80)
for pw, ent in passwords:
    t = format_time(crack_time(ent))
    print(f'{pw:<32}{ent:<18.1f}{t}')

print()
print('Вывод: пароль длиной 12+ символов с большим алфавитом не взломать за разумное время.')
print('WPA3 решает эту проблему архитектурно — офлайн-перебор невозможен в принципе.')

**Разбор результата.** Пароль `12345678` взламывается за секунды. `MyWifi2024` — за ~3 года. `correct-horse-battery-staple` — за миллиарды лет. Энтропия (мера случайности) — главный фактор стойкости. WPA3 с SAE делает офлайн-перебор невозможным, защищая даже слабые пароли.

## Шаг 9. Практический эксперимент

Сравним стойкость разных стратегий выбора пароля. Меняйте параметры и смотрите, как меняется время взлома.

In [ ]:
import math

def entropy_bits(length: int, alphabet_size: int) -> float:
    return length * math.log2(alphabet_size)

def crack_seconds(ent_bits: float, rate: int = 2_500_000) -> float:
    return (2 ** ent_bits) / (2 * rate)

# === ЭКСПЕРИМЕНТ: меняйте длину и алфавит ===
length = 10  # попробуйте 6, 8, 10, 12, 16, 20
alphabet_size = 62  # 26+26+10 (lower+upper+digits). 95 = + спецсимволы

ent = entropy_bits(length, alphabet_size)
sec = crack_seconds(ent)
years = sec / (365 * 86400)

print(f'Длина пароля: {length} символов')
print(f'Размер алфавита: {alphabet_size}')
print(f'Энтропия: {ent:.1f} бит')
if years < 1:
    print(f'Время взлома: {sec/3600:.2f} часов')
elif years < 1e6:
    print(f'Время взлома: {years:.2f} лет')
else:
    print(f'Время взлома: {years:.2e} лет')

print()
print('Сравнение стратегий:')
strategies = [
    ('8 цифр (PIN)', 8, 10),
    ('8 букв lowercase', 8, 26),
    ('10 символов (a-z, A-Z, 0-9)', 10, 62),
    ('12 символов (все 95)', 12, 95),
    ('4 слова из словаря 2048', 4, 2048),  # подход Diceware
    ('16 символов (все 95)', 16, 95),
]
print(f"{'Стратегия':<35}{'Энтропия':<12}{'Время взлома'}")
print('-' * 75)
for name, l, a in strategies:
    e = entropy_bits(l, a)
    y = crack_seconds(e) / (365 * 86400)
    if y < 1:
        t = f'{crack_seconds(e)/3600:.2f} ч'
    elif y < 1e6:
        t = f'{y:.1f} лет'
    else:
        t = f'{y:.2e} лет'
    print(f'{name:<35}{e:<12.1f}{t}')

## 🧠 Проверка понимания

**1. (Объяснение)** Почему WEP считается «взломанным»? Какая именно уязвимость в нём используется?

**2. (Прогноз)** Если перехватчик записал 4-way handshake, сможет ли он прочитать трафик, не зная пароля?

**3. (Объяснение)** Почему WPA3 с SAE защищён от офлайн-перебора, а WPA2-PSK — нет?

---

## ⚠️ Частые ошибки

### Ошибка 1

**Неправильное рассуждение:** «Если у меня стоит WPA2, моя сеть на 100% безопасна».

**Причина ошибки:** Игнорируется возможность офлайн-перебора пароля.

**Правильное объяснение:** WPA2 безопасен только если пароль достаточно длинный и случайный. При слабом пароле атакующий перехватывает handshake и подбирает пароль офлайн.

### Ошибка 2

**Неправильное рассуждение:** «Скрытый SSID (не вещать имя сети) делает сеть невидимой и безопасной».

**Причина ошибки:** Путают невидимость для обычного пользователя с невидимостью для атакующего.

**Правильное объяснение:** Скрытый SSID лишь убирает имя из beacon-кадров, но probe-запросы и ответы видны в эфире. Атакующий с Wireshark найдёт сеть за минуту.

### Ошибка 3

**Неправильное рассуждение:** «WPS упрощает жизнь и не снижает безопасность».

**Причина ошибки:** Не знают об уязвимости Pixie Dust и brute-force PIN.

**Правильное объяснение:** WPS с PIN-кодом имеет критическую уязвимость: 8-значный PIN можно подобрать за часы. WPS нужно отключать в настройках роутера.

### Ошибка 4

**Неправильное рассуждение:** «Открытая сеть в кафе безопасна, если я захожу на HTTPS-сайты».

**Причина ошибки:** Доверяют HTTPS как панацее.

**Правильное объяснение:** HTTPS защищает содержимое, но не метаданные. Атакующий видит, к каким доменам вы обращаетесь (SNI). Возможны downgrade-атаки. Нужен VPN в открытых сетях.

---

## 🎯 Краткий вывод

Безопасность Wi-Fi эволюционировала от уязвимого WEP к современному WPA3. WPA2-PSK всё ещё широко используется, но требует длинного случайного пароля (12+ символов), иначе уязвим к офлайн-перебору через 4-way handshake. WPA3 с SAE закрывает эту проблему архитектурно. Дополнительно нужно: отключать WPS, не полагаться на скрытый SSID, использовать VPN в открытых сетях.

> 😄 Пароль `12345678` на Wi-Fi — это как замок из картонки на двери сейфа: выглядит как замок, но на самом деле просто декорация.

---


## 🔗 Связь между темами

### Какие знания были получены

- Эволюцию протоколов безопасности: WEP → WPA → WPA2 → WPA3
- Как работает 4-way handshake и зачем нужны PMK, PTK, GTK
- Важность энтропии пароля и почему WPS нужно отключать

### Какие знания понадобятся далее

- Понимание того, как настраивать роутер на практике
- Знание про каналы, TX power, ширину канала, DHCP
- Понимание роуминга между точками доступа

**Переход к следующей теме:** В теме 5 мы перейдём от теории к практике: как настроить роутер так, чтобы сеть была и быстрой, и безопасной, и покрывала весь дом. Разберём выбор канала, ширину канала, мощность передатчика и роуминг.

---

## ✅ Мини-проверка

**Вопрос 1** (объяснить причину):

Почему WPA3 устойчив к офлайн-перебору пароля, а WPA2 — нет?

**Вопрос 2** (предсказать результат):

Если пароль — 8 цифр, сколько времени займёт взлом WPA2 на одной RTX 4090?

**Вопрос 3** (выбрать правильный вариант):

Что НЕ является частью PTK? (a) KCK (b) KEK (c) TK (d) GTK

**Вопрос 4** (найти ошибку):

«Я скрыл SSID, теперь мою сеть никто не найдёт». Что не так?

---

## 🔬 Мини-лабораторная

**Цель:** Научиться оценивать стойкость пароля через вычисление энтропии и времени взлома.

### Пошаговая инструкция

1. Придумайте 5 разных паролей разной сложности (от простого к сложному).
2. Для каждого посчитайте энтропию: длина × log2(размер_алфавита).
3. Используя скорость перебора 2.5 млн PMK/сек, оцените время взлома.
4. Постройте логарифмический график: по оси X — длина пароля, по оси Y — время взлома.
5. Найдите минимальную длину пароля, при которой взлом занимает более 100 лет.

**Ожидаемый результат:** График с экспоненциальным ростом времени взлома от длины. Минимум — около 12 символов при смешанном алфавите.

**Объяснение результата:** Эта лабораторная даёт интуицию: добавление 1 символа к паролю увеличивает время взлома в `размер_алфавита` раз. Поэтому 16-значный пароль уже практически не взломать.

---

## 🧪 Эксперимент

**Задание:** В коде Шага 9 меняйте параметры длины и размера алфавита. Найдите, при какой длине пароль из 4 слов словаря 2048 (как Diceware) даёт энтропию 44 бита?

Изменяемые параметры:

- length = 1, 2, 3, 4, 5, 6 (для словарных паролей)
- alphabet_size = 2048 (словарь Diceware)

**Что наблюдаем:** 4 слова × log2(2048) = 4 × 11 = 44 бита. Это уже неплохо, а 6 слов = 66 бит — стойко даже против государственных атакующих.

---

## 📝 Тестовые задания с вариантами ответов

> Выберите один правильный вариант в каждом задании. Ответы — в конце блока.

### Задание 1

Какой протокол Wi-Fi считается устаревшим и не должен использоваться?

- [ ] **a)** WPA3-Personal
- [ ] **b)** WPA2-PSK
- [ ] **c)** WEP
- [ ] **d)** WPA2-Enterprise

### Задание 2

Что такое 4-way handshake?

- [ ] **a)** Процедура отключения клиента от сети
- [ ] **b)** Обмен 4 сообщениями между клиентом и AP для выработки PTK
- [ ] **c)** Алгоритм шифрования трафика
- [ ] **d)** Метод сжатия Wi-Fi-кадров

### Задание 3

Какая уязвимость есть в WPS?

- [ ] **a)** WPS не шифрует трафик
- [ ] **b)** 8-значный PIN можно подобрать за разумное время
- [ ] **c)** WPS использует слабый RC4
- [ ] **d)** WPS передаёт пароль в открытом виде

### Задание 4

WPA3 использует для аутентификации:

- [ ] **a)** PSK
- [ ] **b)** SAE
- [ ] **c)** RC4
- [ ] **d)** WEP-PIN

### Задание 5

Какой пароль наиболее стойкий к взлому WPA2?

- [ ] **a)** 12345678
- [ ] **b)** password1234
- [ ] **c)** MyDog2024
- [ ] **d)** Tr0ub4dour&3-correct-horse

<details><summary><b>🔑 Ответы и пояснения (нажмите, чтобы развернуть)</b></summary>

**Задание 1:** правильный ответ — **c)**. WEP взламывается за минуты из-за слабого RC4 и короткого ключа. Не используйте WEP никогда.

**Задание 2:** правильный ответ — **b)**. 4-way handshake — это 4 сообщения (M1-M4) между клиентом и AP, в ходе которых обе стороны доказывают знание PMK и вырабатывают общий PTK.

**Задание 3:** правильный ответ — **b)**. PIN WPS делится на две половины (4+3 цифры), что снижает пространство перебора с 10⁸ до 10⁴+10³. Это даёт атаку brute-force за часы.

**Задание 4:** правильный ответ — **b)**. SAE (Simultaneous Authentication of Equals) — протокол Dragonfly, устойчивый к офлайн-перебору. Это главное архитектурное улучшение WPA3.

**Задание 5:** правильный ответ — **d)**. Последний вариант имеет ~100 бит энтропии, что делает взлом невозможным за разумное время. Остальные взламываются за минуты или дни.

</details>

---

## ✅ Чек-лист темы

- Я понимаю эволюцию протоколов безопасности Wi-Fi.
- Я понимаю, как работает 4-way handshake.
- Я умею оценивать стойкость пароля через энтропию.
- Я умею реализовать AES-шифрование на Python через библиотеку cryptography.
- Я могу объяснить, почему WPS нужно отключать.
- Я могу объяснить, чем WPA3 лучше WPA2.

---

# 📘 Тема 5. Настройка Wi-Fi сети: каналы, мощность, DHCP, роуминг

## Пять вопросов темы

| Вопрос | Ответ |
|---|---|
| **Что это?** | Настройка Wi-Fi — это выбор параметров точки доступа: канала, ширины канала, мощности передатчика, SSID, пароля, DHCP-пула, режима роуминга. |
| **Зачем существует?** | Без грамотной настройки сеть будет медленной, нестабильной, с «мёртвыми зонами». Соседские роутеры будут мешать друг другу, а устройства — терять сеть при перемещении по дому. |
| **Как работает?** | Через веб-интерфейс роутера (обычно 192.168.1.1) или протоколы управления (TR-069, Mesh-системы). Настраиваются: радиопараметры, DHCP, безопасность, гостевая сеть, роуминг. |
| **Почему именно так?** | Потому что радиоспектр — общий ресурс, и оптимальные параметры зависят от окружения: стен, соседей, количества устройств, их типа. |
| **Где применяется?** | Дома (один роутер или mesh), в офисе (несколько AP с роумингом), на предприятии (контроллер + десятки AP), в кафе (гостевая сеть). |

---

## Шаг 1. Постановка проблемы

Вы купили новый роутер, принесли домой, подключили к розетке и провайдеру. Сеть появилась. Скорость — 600 Мбит/с рядом с роутером, но в дальней спальне — 20 Мбит/с. Видео в YouTube грузится, голос в Zoom дёргается, а умный чайник на кухне вообще не подключается.

Знакомая ситуация? Это типичные последствия дефолтной настройки:

- Роутер стоит на канале 6 — там же 5 соседей.
- Ширина канала 40 МГц в 2,4 ГГц — занимает половину диапазона.
- Мощность 100% — ревёт на весь подъезд, ловит помехи от всех.
- 5 ГГц отключён по умолчанию (старые прошивки).
- DHCP раздаёт адреса с 192.168.0.100 — конфликтует с соседской сетью.

**Проблема:** как настроить роутер так, чтобы скорость была высокой во всём доме, не было помех от соседей, и устройства переключались между точками доступа без обрыва?

**Решение:** грамотно выбрать канал, ширину канала, мощность, диапазоны, настроить DHCP и роуминг.

## Шаг 2. Практический пример

Смоделируем «тепловую карту» покрытия Wi-Fi в квартире. Это типичная задача при планировании сети: где поставить роутер, нужны ли дополнительные точки.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Поиск шрифта с поддержкой кириллицы по нескольким известным путям
_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf',
    '/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]
for _p in _FONT_CANDIDATES:
    if os.path.exists(_p):
        try:
            fm.fontManager.addfont(_p)
        except Exception:
            pass
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Liberation Sans', 'FreeSans']
plt.rcParams['axes.unicode_minus'] = False

def signal_dbm(tx_power_dbm: float, distance_m: float, freq_ghz: float,
               walls: list[tuple[float, float]] = None) -> float:
    """Упрощённая модель затухания сигнала с учётом стен."""
    # FSPL в дБ
    if distance_m < 0.5:
        distance_m = 0.5
    fspl = 20 * np.log10(distance_m) + 20 * np.log10(freq_ghz * 1e9) - 147.55
    walls_loss = sum(loss for _, loss in walls) if walls else 0
    return tx_power_dbm - fspl - walls_loss

def coverage_heatmap(router_pos, tx_power=20, freq_ghz=2.4,
                     walls=None, grid_size=(100, 80), room_size=(10, 8)):
    """Возвращает 2D-массив уровней сигнала в дБм."""
    xs = np.linspace(0, room_size[0], grid_size[0])
    ys = np.linspace(0, room_size[1], grid_size[1])
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(grid_size[1]):
        for j in range(grid_size[0]):
            d = np.sqrt((X[i,j] - router_pos[0])**2 + (Y[i,j] - router_pos[1])**2)
            # Подсчёт числа стен между роутером и точкой
            walls_between = 0
            if walls:
                for wx, wloss in walls:
                    if router_pos[0] < wx < X[i,j] or X[i,j] < wx < router_pos[0]:
                        walls_between += wloss
            Z[i,j] = signal_dbm(tx_power, d, freq_ghz, [(0, walls_between)] if walls_between else None)
    return X, Y, Z

# Роутер в углу квартиры 10x8 м
X, Y, Z = coverage_heatmap(
    router_pos=(1, 1),
    tx_power=20,   # дБм — типичная мощность роутера
    freq_ghz=2.4,
    walls=[(5, 8), (5, 8)],  # две стены по 8 дБ потерь
    room_size=(10, 8)
)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
levels = np.arange(-90, -30, 5)
cs = ax.contourf(X, Y, Z, levels=levels, cmap='RdYlGn')
ax.plot(1, 1, 'k^', markersize=15, label='Роутер')
ax.set_title('Покрытие Wi-Fi 2,4 ГГц в квартире 10×8 м', fontsize=13, fontweight='bold')
ax.set_xlabel('X, м')
ax.set_ylabel('Y, м')
cbar = fig.colorbar(cs, ax=ax, label='Уровень сигнала, дБм')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.show()

**Разбор результата.** Красно-жёлто-зелёная карта показывает уровень сигнала в каждой точке квартиры. Зелёный — сильный сигнал (>-60 дБм), жёлтый — средний (-60..-75), красный — слабый (<-75, плохо работает). Видно, что две внутренние стены по 8 дБ потерь создают «мёртвую зону» в дальнем углу. Именно поэтому роутер лучше ставить в центре квартиры.

## Шаг 3. Интуитивное объяснение

Настройка Wi-Fi похожа на настройку радио в машине:

- **Канал** — это радиочастота, на которой работает сеть. Нужно выбрать «свободную волну».
- **Мощность** — это громкость. Слишком тихо — плохо слышно. Слишком громко — соседи будут жаловаться.
- **Ширина канала** — это «полоса дороги». 20 МГц — узкая, но стабильно работает. 160 МГц — широкая, быстро, но мешает соседям.
- **DHCP** — это «выдача адресов». Каждое устройство получает свой IP-адрес автоматически.
- **Роуминг** — это «передача клиента» между AP при перемещении по дому/офису.

## Шаг 4. Жизненная аналогия

Представьте отель с 100 номерами:

- **DHCP** — это ресепшен, который выдаёт ключи от номеров (IP-адреса) гостям. Если ресепшен не работает — гость стоит в холле и не знает, куда идти.
- **Каналы Wi-Fi** — это этажи отеля. Если все гости живут на одном этаже (канал 6), лифты переполнены, лифт остановился на каждом этаже. Лучше распределить по этажам (1, 6, 11).
- **Мощность передатчика** — это громкость радио в номере. Сосед через стену не должен слышать ваше радио.
- **Роуминг** — это когда гость переходит из одного корпуса отеля в другой без переселения: ключ продолжает работать, багаж не нужно тащить обратно на ресепшен.

## Шаг 5. Техническое объяснение

Ключевые параметры настройки роутера:

| Параметр | Что делает | Типичные значения |
|---|---|---|
| **SSID** | Имя сети | Любое, лучше без личных данных |
| **Канал (2,4 ГГц)** | Частотный канал | 1, 6 или 11 (или Auto) |
| **Ширина канала (2,4 ГГц)** | Полоса частот | 20 МГц (стабильно) |
| **Канал (5 ГГц)** | Частотный канал | Любой непересекающийся |
| **Ширина (5 ГГц)** | Полоса | 80 МГц (баланс) или 160 МГц |
| **TX Power** | Мощность передатчика | 50–100% (или по обстановке) |
| **Band Steering** | Перевод устройств на 5 ГГц | Включить |
| **WPA3 / WPA2** | Шифрование | WPA3, если поддерживается |
| **DHCP pool** | Диапазон IP | 192.168.1.100–192.168.1.200 |
| **Роуминг 802.11r/k/v** | Быстрое переключение AP | Включить в mesh |

**Band Steering** — функция, которая заставляет устройства подключаться к 5 ГГц вместо 2,4 ГГц, если они поддерживают оба диапазона. Освобождает 2,4 ГГц для IoT-устройств и старой техники.

**DHCP (Dynamic Host Configuration Protocol)** — протокол, по которому роутер автоматически выдаёт устройствам IP-адреса, маску подсети, шлюз и DNS. Без DHCP каждое устройство пришлось бы настраивать вручную.

## Шаг 6. Внутреннее устройство

Как роутер обрабатывает подключение нового устройства:

```mermaid
sequenceDiagram
    participant D as Устройство (телефон)
    participant A as Роутер (AP+DHCP)
    participant I as Интернет
    
    D->>A: Probe Request (кто тут AP?)
    A->>D: Probe Response (я AP, SSID=Home)
    D->>A: Authentication Request
    A->>D: Authentication Response
    D->>A: Association Request (хочу подключиться)
    A->>D: Association Response (OK, AID=1)
    Note over D,A: 4-way handshake (установка ключей WPA)
    D->>A: DHCP Discover (дайте IP!)
    A->>D: DHCP Offer (192.168.1.105?)
    D->>A: DHCP Request (беру 192.168.1.105)
    A->>D: DHCP ACK (подтверждаю, на 24 часа)
    D->>I: HTTP-запрос через шлюз
    I->>D: HTTP-ответ
```

После DHCP-коммуникации устройство получает:
- IP-адрес (например, 192.168.1.105)
- Маску подсети (255.255.255.0)
- Шлюз по умолчанию (192.168.1.1 — сам роутер)
- DNS-сервер (обычно тоже роутер или 8.8.8.8)
- Срок аренды (24 часа — после чего нужно продлить)

## Шаг 7. Визуальная схема

Топология типичной домашней сети:

```mermaid
flowchart TB
    INET[Интернет<br/>провайдер] -->|оптика/кабель| MOD[ONT/модем]
    MOD -->|Ethernet| RT[Роутер<br/>192.168.1.1]
    RT -->|Wi-Fi 2.4 ГГц| D1[Телефон]
    RT -->|Wi-Fi 5 ГГц| D2[Ноутбук]
    RT -->|Wi-Fi 2.4 ГГц| D3[Умный чайник]
    RT -->|Ethernet| D4[Смарт ТВ]
    RT -->|Wi-Fi 6| D5[Игровой ПК]
    RT -.гостевая.-> G[Гостевая сеть<br/>изолированная]
    RT -->|DNS, DHCP, NAT| INTERNAL[Локальная подсеть<br/>192.168.1.0/24]
    style RT fill:#bfdbfe
    style INET fill:#e0e7ff
    style G fill:#fef3c7
```

Mesh-система с роумингом (для больших домов):

```mermaid
flowchart LR
    INET[Интернет] --> R1[Главный роутер<br/>192.168.1.1]
    R1 <-.mesh-линк 5 ГГц.-> R2[Точка 2<br/>192.168.1.2]
    R2 <-.mesh-линк.-> R3[Точка 3<br/>192.168.1.3]
    D1[Телефон] -.роуминг.-> R1
    D1 -.роуминг.-> R2
    D1 -.роуминг.-> R3
    style R1 fill:#bfdbfe
    style R2 fill:#c7d2fe
    style R3 fill:#c7d2fe
```

## Шаг 8. Рабочий пример

Симуляция DHCP-сервера. Покажем, как роутер раздаёт IP-адреса из пула.

In [ ]:
import random
from dataclasses import dataclass, field
from datetime import datetime, timedelta

@dataclass
class DHCPLease:
    ip: str
    mac: str
    hostname: str
    leased_at: datetime
    lease_duration: timedelta = field(default_factory=lambda: timedelta(hours=24))
    
    def expires_at(self) -> datetime:
        return self.leased_at + self.lease_duration

class DHCPServer:
    def __init__(self, pool_start: str, pool_end: str):
        # Разбор пула: 192.168.1.100 -> 192.168.1.200
        prefix = '.'.join(pool_start.split('.')[:3])
        start = int(pool_start.split('.')[3])
        end = int(pool_end.split('.')[3])
        self.available = [f'{prefix}.{i}' for i in range(start, end + 1)]
        self.leases: dict[str, DHCPLease] = {}  # mac -> lease
    
    def request_ip(self, mac: str, hostname: str) -> str:
        # Если уже есть аренда — продлеваем
        if mac in self.leases:
            self.leases[mac].leased_at = datetime.now()
            return self.leases[mac].ip
        if not self.available:
            return None
        ip = self.available.pop(0)
        self.leases[mac] = DHCPLease(ip, mac, hostname, datetime.now())
        return ip
    
    def show_leases(self):
        print(f"{'IP':<18}{'MAC':<20}{'Hostname':<15}{'Истекает'}")
        print('-' * 75)
        for lease in self.leases.values():
            print(f'{lease.ip:<18}{lease.mac:<20}{lease.hostname:<15}'
                  f'{lease.expires_at().strftime("%Y-%m-%d %H:%M")}')

# Симулируем роутер с пулом 192.168.1.100-200
dhcp = DHCPServer('192.168.1.100', '192.168.1.200')

# Подключаем несколько устройств
devices = [
    ('AA:BB:CC:11:22:33', 'iPhone-Anna'),
    ('AA:BB:CC:44:55:66', 'MacBook-Pro'),
    ('AA:BB:CC:77:88:99', 'Smart-TV-Sony'),
    ('AA:BB:CC:AA:BB:CC', 'IoT-Kettle'),
    ('AA:BB:CC:DD:EE:FF', 'Pixel-8'),
]
for mac, host in devices:
    ip = dhcp.request_ip(mac, host)
    print(f'Выдан IP {ip} устройству {host} (MAC {mac})')

print()
print('Таблица аренды DHCP:')
dhcp.show_leases()

**Разбор результата.** DHCP-сервер выдаёт каждому устройству IP-адрес из пула. Адреса уникальны в пределах подсети. Срок аренды — 24 часа: если устройство не вышло на связь за это время, адрес возвращается в пул. Это предотвращает исчерпание пула при большом числе гостей.

## Шаг 9. Практический эксперимент

Исследуем влияние мощности передатчика (TX Power) на покрытие. Меняйте `tx_power` и смотрите, как меняется «полезная зона».

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Поиск шрифта с поддержкой кириллицы по нескольким известным путям
_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf',
    '/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]
for _p in _FONT_CANDIDATES:
    if os.path.exists(_p):
        try:
            fm.fontManager.addfont(_p)
        except Exception:
            pass
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Liberation Sans', 'FreeSans']
plt.rcParams['axes.unicode_minus'] = False

# === ЭКСПЕРИМЕНТ: меняйте TX power ===
tx_powers = [10, 17, 20, 23, 30]  # дБм

fig, axes = plt.subplots(1, len(tx_powers), figsize=(20, 4), constrained_layout=True)

for ax, tx in zip(axes, tx_powers):
    # Расчёт радиуса покрытия при пороге -75 дБм
    # FSPL = tx - (-75) = tx + 75
    # 20*log10(d) + 20*log10(2.4e9) - 147.55 = tx + 75
    # d = 10^((tx + 75 - 20*log10(2.4e9) + 147.55) / 20)
    max_loss = tx + 75
    d_max = 10 ** ((max_loss - 20 * np.log10(2.4e9) + 147.55) / 20)
    
    # Тепловая карта
    xs = np.linspace(-15, 15, 200)
    ys = np.linspace(-10, 10, 200)
    X, Y = np.meshgrid(xs, ys)
    D = np.sqrt(X**2 + Y**2)
    signal = tx - (20 * np.log10(np.maximum(D, 0.5)) + 20 * np.log10(2.4e9) - 147.55)
    
    levels = np.arange(-95, -30, 5)
    ax.contourf(X, Y, signal, levels=levels, cmap='RdYlGn')
    ax.plot(0, 0, 'k^', markersize=12)
    ax.set_title(f'TX = {tx} дБм\nрадиус ≈ {d_max:.1f} м', fontsize=11)
    ax.set_xlim(-15, 15)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Влияние TX Power на радиус покрытия (порог -75 дБм)', fontsize=14, fontweight='bold')
plt.show()

print('Вывод: каждые +3 дБм мощности удваивают радиус покрытия.')
print('Но! В РФ максимальная EIRP для Wi-Fi 2.4 ГГц — 100 мВт (20 дБм).')
print('Превышать не только незаконно, но и бессмысленно: устройства всё равно не «докричатся» обратно.')

**Что наблюдаем.** При TX=10 дБм радиус покрытия ~3 м (одна комната). При TX=20 дБм — ~10 м (квартира). При TX=30 дБм — ~30 м (но это уже превышение лимитов).

Важно понимать: TX Power роутера ничего не значит, если телефон не может «докричаться» обратно. Телефон имеет TX Power ~15 дБм — слабее роутера. Поэтому повышение мощности роутера свыше 20 дБм не помогает: вы слышите телефон, а он вас — нет.

## 🧠 Проверка понимания

**1. (Объяснение)** Почему повышение TX Power роутера выше 20 дБм не улучшает связь с телефоном?

**2. (Прогноз)** Если в 2,4 ГГц все соседи включат ширину канала 40 МГц, что произойдёт?

**3. (Объяснение)** Зачем нужен роуминг 802.11r/k/v и чем отличаются эти три протокола?

---

## ⚠️ Частые ошибки

### Ошибка 1

**Неправильное рассуждение:** «Поставлю мощность роутера на максимум — будет лучше везде».

**Причина ошибки:** Не учитывается асимметрия: телефон слабее роутера.

**Правильное объяснение:** Если роутер «кричит» на 30 дБм, а телефон на 15, вы слышите роутер издалека, но он вас — нет. Связь не установится. Лучше уменьшить TX Power роутера, чтобы он не «звал» устройства из зон, где они не смогут ответить.

### Ошибка 2

**Неправильное рассуждение:** «Включу 40 МГц в 2,4 ГГц — будет в 2 раза быстрее».

**Причина ошибки:** Теоретически да, на практике — нет.

**Правильное объяснение:** 40 МГц в 2,4 ГГц занимает 2/3 всего диапазона. Любой сосед на каналах 1-11 будет вам мешать. Реальная скорость упадёт. В 2,4 ГГц всегда используйте только 20 МГц.

### Ошибка 3

**Неправильное рассуждение:** «Роутер поставлю в углу за телевизором — там розетка удобная».

**Причина ошибки:** Выбор места по удобству, а не по физике сигнала.

**Правильное объяснение:** Роутер нужно ставить в центре квартиры, в открытом месте, выше уровня мебели. Угол и металлический корпус ТВ создают мёртвые зоны. Лучше потерять 30 минут на перестановку, чем год мучиться со слабым сигналом.

### Ошибка 4

**Неправильное рассуждение:** «DHCP пул сделаю 192.168.1.1-192.168.1.255 — хватит всем».

**Причина ошибки:** Путают диапазон IP с подсетью.

**Правильное объяснение:** 192.168.1.0 — сетевой адрес, 192.168.1.255 — широковещательный. Их нельзя выдавать устройствам. Пул должен быть, например, 192.168.1.100-192.168.1.200. Хватит для 100 устройств.

---

## 🎯 Краткий вывод

Грамотная настройка Wi-Fi: выбрать непересекающийся канал в 2,4 ГГц (1, 6 или 11), ширину 20 МГц в 2,4 и 80 МГц в 5 ГГц, мощность 17–20 дБм (не выше!), включить WPA3, Band Steering, отключить WPS. DHCP-пул — разумного размера (50-100 адресов). Для больших квартир — mesh-система с роумингом 802.11r/k/v.

> 😄 Роутер за телевизором в углу квартиры — это как микрофон в шкафу на сцене: поёт, но никто не слышит.

---


## 🔗 Связь между темами

### Какие знания были получены

- Как выбрать канал, ширину канала, TX Power
- Как работает DHCP и зачем нужен пул адресов
- Что такое Band Steering и роуминг 802.11r/k/v
- Как планировать покрытие через тепловую карту

### Какие знания понадобятся далее

- Понимание метрик качества сети: RSSI, SNR, MCS index
- Умение диагностировать медленный интернет
- Знание про Channel Utilization, retry rate, помехи

**Переход к следующей теме:** В теме 6 мы разберём, как диагностировать проблемы уже настроенной сети: почему интернет «тормозит», как найти источник помех, как измерить реальную скорость и что делать, если не помогает ничего.

---

## ✅ Мини-проверка

**Вопрос 1** (объяснить причину):

Почему в 2,4 ГГц нельзя ставить ширину канала 40 МГц в многоквартирном доме?

**Вопрос 2** (предсказать результат):

Если поднять TX Power роутера с 20 до 30 дБм, улучшится ли связь с телефоном в дальней комнате?

**Вопрос 3** (выбрать правильный вариант):

Какой пул DHCP корректен для подсети /24 (255.255.255.0)? (a) 192.168.1.0-192.168.1.255 (b) 192.168.1.100-192.168.1.200 (c) 192.168.1.255 (d) 0.0.0.0-255.255.255.255

**Вопрос 4** (объяснить последовательность):

Опишите порядок настройки нового роутера: SSID, пароль, канал, DHCP — в каком порядке и почему?

---

## 🔬 Мини-лабораторная

**Цель:** Спроектировать оптимальное покрытие Wi-Fi в квартире 12×10 м с двумя внутренними стенами.

### Пошаговая инструкция

1. Используйте функцию coverage_heatmap из Шага 2.
2. Разместите роутер в 4 разных позициях: угол (1,1), центр (6,5), у окна (10,5), в коридоре (3,8).
3. Для каждого варианта постройте тепловую карту.
4. Посчитайте площадь зоны с сигналом лучше -70 дБм.
5. Выберите лучшую позицию и объясните почему.

**Ожидаемый результат:** 4 тепловые карты. Лучший вариант — центр квартиры (6,5), так как даёт максимальную «зелёную» зону.

**Объяснение результата:** Эта лабораторная показывает: позиция роутера важнее его мощности. Центральное расположение всегда даёт лучшее покрытие, чем угловое, потому что сигнал расходится радиально.

---

## 🧪 Эксперимент

**Задание:** В коде Шага 9 меняйте TX Power и порог «полезного сигнала». Найдите, при каком TX Power радиус покрытия при пороге -75 дБм достигает 15 метров.

Изменяемые параметры:

- tx_power: 5, 10, 15, 20, 25, 30
- Порог: попробуйте -65 (отличный), -75 (хороший), -85 (минимальный)

**Что наблюдаем:** Для 15 м при пороге -75 дБм нужно TX ≈ 24 дБм. Но это уже выше легального лимита в РФ для 2,4 ГГц (20 дБм). Вывод: нельзя «пробить» 15 м одной AP — нужна mesh-система.

---

## 📝 Тестовые задания с вариантами ответов

> Выберите один правильный вариант в каждом задании. Ответы — в конце блока.

### Задание 1

Какая ширина канала рекомендована для 2,4 ГГц в многоквартирном доме?

- [ ] **a)** 20 МГц
- [ ] **b)** 40 МГц
- [ ] **c)** 80 МГц
- [ ] **d)** 160 МГц

### Задание 2

Что такое Band Steering?

- [ ] **a)** Шифрование трафика между AP и клиентом
- [ ] **b)** Автоматический перевод устройств на 5 ГГц
- [ ] **c)** Управление мощностью передатчика
- [ ] **d)** Протокол роуминга

### Задание 3

Какой IP нельзя выдавать устройствам в подсети 192.168.1.0/24?

- [ ] **a)** 192.168.1.50
- [ ] **b)** 192.168.1.100
- [ ] **c)** 192.168.1.255
- [ ] **d)** 192.168.1.200

### Задание 4

Что делает 802.11r (Fast Transition)?

- [ ] **a)** Шифрует трафик сильнее
- [ ] **b)** Ускоряет роуминг между AP
- [ ] **c)** Увеличивает TX Power
- [ ] **d)** Переводит устройства в 5 ГГц

### Задание 5

Почему повышение TX Power роутера выше 20 дБм не помогает связи?

- [ ] **a)** Это незаконно
- [ ] **b)** Телефон не может «докричаться» обратно
- [ ] **c)** Устройства не поддерживают такую мощность
- [ ] **d)** Помехи от соседей усиливаются

<details><summary><b>🔑 Ответы и пояснения (нажмите, чтобы развернуть)</b></summary>

**Задание 1:** правильный ответ — **a)**. В 2,4 ГГц нужно использовать только 20 МГц — иначе сеть займёт почти весь диапазон и будет мешать соседям.

**Задание 2:** правильный ответ — **b)**. Band Steering — функция роутера, которая «стимулирует» двухдиапазонные устройства подключаться к 5 ГГц, освобождая 2,4 ГГц для IoT.

**Задание 3:** правильный ответ — **c)**. 192.168.1.255 — широковещательный адрес подсети /24. Его нельзя назначать никакому устройству.

**Задание 4:** правильный ответ — **b)**. 802.11r (Fast BSS Transition) сокращает время 4-way handshake при роуминге с 100+ мс до <50 мс. Критично для голосовых звонков.

**Задание 5:** правильный ответ — **b)**. Wi-Fi — двунаправленная связь. Если роутер «слышит» телефон издалека, но телефон не слышит роутер — связи нет. TX Power клиента (телефона) — ограничение батареей, обычно 15 дБм.

</details>

---

## ✅ Чек-лист темы

- Я понимаю, как выбрать канал и ширину канала для 2,4 и 5 ГГц.
- Я понимаю, как работает DHCP и какие адреса нельзя выдавать.
- Я умею строить тепловую карту покрытия через matplotlib.
- Я умею симулировать DHCP-сервер на Python.
- Я могу объяснить, почему TX Power выше 20 дБм бесполезен.
- Я могу объяснить, что такое Band Steering и роуминг 802.11r/k/v.

---

# 📘 Тема 6. Анализ и оптимизация Wi-Fi: диагностика, помехи, скорость

## Пять вопросов темы

| Вопрос | Ответ |
|---|---|
| **Что это?** | Анализ Wi-Fi — это измерение метрик сети (RSSI, SNR, Channel Utilization, скорость, retry rate) и поиск причин медленной или нестабильной работы. |
| **Зачем существует?** | Даже правильно настроенная сеть деградирует со временем: появляются новые соседи, ломается кабель, обновляется прошивка. Без диагностики невозможно понять, что именно тормозит. |
| **Как работает?** | Через встроенные утилиты ОС (ping, traceroute, netsh, airport), специализированные приложения (WiFi Analyzer, Ekahau, NetSpot) и анализ логов роутера. |
| **Почему именно так?** | Потому что видимые симптомы («интернет тормозит») имеют много причин: помехи, слабый сигнал, перегрузка канала, проблема с провайдером, DNS-проблемы. Нужна методика. |
| **Где применяется?** | Дома при жалобах «медленный интернет», в офисе при планировании.capacity, в кафе при плохих отзывах гостей. |

---

## Шаг 1. Постановка проблемы

Пользователь жалуется: «Интернет еле работает, YouTube грузится, Zoom обрывается». На вопрос «что не так?» — пожимает плечами. Возможных причин десяток:

- Слабый сигнал в дальней комнате.
- Помехи от соседского роутера на том же канале.
- Перегрузка канала (Channel Utilization > 70%).
- Старый роутер без поддержки 5 ГГц.
- Проблема у провайдера ( loss на линии).
- DNS-сервер провайдера тормозит.
- Устройство пользователя устарело (Wi-Fi 4 в 2024).
- Сосед включил микроволновку (помеха на 2,4 ГГц).
- Bluetooth-наушники мешают (тоже 2,4 ГГц).
- Металлический шкаф встал между роутером и диваном.

**Проблема:** как за 15 минут понять, что именно не так, и что чинить?

**Решение:** систематическая диагностика — от измерения базовых метрик (RSSI, SNR, скорость) до анализа канала и изоляции источника проблемы.

## Шаг 2. Практический пример

Измерим «скорость интернета» прямо в Colab. Мы не можем измерить скорость Wi-Fi (нет адаптера), но можем измерить скорость от Colab-сервера до тестового сервера в интернете — это покажет, как вообще работает измерение пропускной способности.

In [ ]:
import time
import urllib.request
import socket

def measure_download_speed(url: str, timeout: float = 10) -> dict:
    """Скачивает файл и измеряет скорость."""
    start = time.time()
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            data = resp.read()
        elapsed = time.time() - start
        size_mb = len(data) / (1024 * 1024)
        speed_mbps = (size_mb * 8) / elapsed if elapsed > 0 else 0
        return {
            'url': url,
            'size_mb': round(size_mb, 2),
            'time_s': round(elapsed, 2),
            'speed_mbps': round(speed_mbps, 2),
            'success': True
        }
    except Exception as e:
        return {'url': url, 'error': str(e), 'success': False}

def measure_latency(host: str = '8.8.8.8', count: int = 5) -> dict:
    """Измеряет задержку через TCP-подключение."""
    latencies = []
    for _ in range(count):
        start = time.time()
        try:
            sock = socket.create_connection((host, 53), timeout=3)
            sock.close()
            latencies.append((time.time() - start) * 1000)
        except Exception:
            pass
        time.sleep(0.2)
    if not latencies:
        return {'host': host, 'error': 'no response'}
    return {
        'host': host,
        'min_ms': round(min(latencies), 1),
        'avg_ms': round(sum(latencies) / len(latencies), 1),
        'max_ms': round(max(latencies), 1),
        'jitter_ms': round(max(latencies) - min(latencies), 1),
    }

# Тест задержки
print('=== Тест задержки (latency) до 8.8.8.8 ===')
lat = measure_latency()
for k, v in lat.items():
    print(f'  {k}: {v}')

print()
print('=== Тест скорости скачивания ===')
# Используем публичные тестовые файлы Cloudflare
test_urls = [
    ('Cloudflare 100KB', 'https://speed.cloudflare.com/__down?bytes=102400'),
    ('Cloudflare 1MB', 'https://speed.cloudflare.com/__down?bytes=1048576'),
    ('Cloudflare 10MB', 'https://speed.cloudflare.com/__down?bytes=10485760'),
]
for name, url in test_urls:
    r = measure_download_speed(url)
    if r['success']:
        print(f'  {name}: {r["size_mb"]} МБ за {r["time_s"]} с = {r["speed_mbps"]} Мбит/с')
    else:
        print(f'  {name}: ошибка {r.get("error")}')

**Разбор результата.** Мы измерили две ключевые метрики:
- **Latency (RTT)** — время прохождения пакета туда-обратно. Норма для Wi-Fi: 1–20 мс, для провайдера: 5–50 мс. Высокий jitter (>30 мс) ломает голосовые звонки.
- **Throughput** — скорость скачивания. Норма для домашнего Wi-Fi 5: 200–500 Мбит/с, для Wi-Fi 6: 500–1500 Мбит/с.

Эти метрики — отправная точка диагностики. Если latency высокая — проблема не в Wi-Fi, а в маршруте. Если скорость низкая при низкой задержке — проблема в Wi-Fi или провайдере.

## Шаг 3. Интуитивное объяснение

Диагностика Wi-Fi похожа на визит к врачу:

1. **Жалоба** — «интернет тормозит». Аналог: «у меня болит голова».
2. **Анамнез** — когда началось? Что менялось? Аналог: «как давно? после чего?»
3. **Анализы** — измерить скорость, latency, RSSI, SNR. Аналог: давление, температура, анализы крови.
4. **Диагноз** — например, «перегружен канал 6». Аналог: «гипертония».
5. **Лечение** — переключить на канал 11. Аналог: «таблетки от давления».
6. **Контроль** — через неделю проверить снова. Аналог: «прийти на повторный приём».

Главное правило: лечить нужно причину, а не симптом. Если интернет медленный из-за помех, покупка «более мощного роутера» не поможет — нужно менять канал.

## Шаг 4. Жизненная аналогия

Wi-Fi-канал — это дорога. Скорость интернета — это скорость движения по этой дороге.

- **RSSI** (уровень сигнала) — это «как хорошо видно дорогу». Если темно — едешь медленно.
- **SNR** (signal-to-noise ratio) — это «насколько дорога свободна от шума». Если шумно — тоже медленно.
- **Channel Utilization** — это «загрузка дороги машинами». Пробка — низкая скорость.
- **Retry rate** — это «сколько раз пришлось переотправить пакет». Аналог: развороты из-за перекрытых улиц.
- **MCS index** — это «скоростной режим». На хорошей дороге можно ехать 130, на плохой — 30.

Если интернет «тормозит» — нужно проверить каждую метрику и найти «узкое горлышко».

## Шаг 5. Техническое объяснение

Ключевые метрики Wi-Fi:

| Метрика | Что измеряет | Норма | Плохо |
|---|---|---|---|
| **RSSI** | Уровень сигнала в дБм | -30 до -60 | < -75 |
| **SNR** | Отношение сигнал/шум в дБ | > 30 | < 15 |
| **Channel Utilization** | Загрузка канала в % | < 50% | > 70% |
| **Retry rate** | Доля переотправленных кадров | < 10% | > 30% |
| **MCS index** | Индекс модуляции (0–11) | 7–11 (Wi-Fi 6) | 0–3 |
| **Latency (RTT)** | Задержка до шлюза | 1–20 мс | > 100 мс |
| **Jitter** | Разброс задержки | < 10 мс | > 30 мс |
| **Throughput** | Реальная скорость | > 50% от теории | < 20% |

**Связь метрик:** MCS index автоматически выбирается на основе SNR. Чем выше SNR, тем более сложную модуляцию можно использовать, тем выше скорость. При SNR < 10 дБ устройство откатывается на самую медленную модуляцию (BPSK 1/2) — это ~6 Мбит/с.

## Шаг 6. Внутреннее устройство

Как MAC-слой выбирает скорость передачи:

```mermaid
flowchart TD
    A[Кадр для отправки] --> B{Текущий MCS?}
    B --> C[Отправка кадра]
    C --> D{Получен ACK?}
    D -- Да --> E[Увеличить MCS<br/>если streak >= 10]
    D -- Нет --> F[Увеличить счётчик потерь]
    F --> G{Потери > порога?}
    G -- Да --> H[Снизить MCS на 1 шаг]
    G -- Нет --> I[Сохранить MCS]
    E --> C
    H --> C
    I --> C
    style E fill:#bbf7d0
    style H fill:#fecaca
```

Это алгоритм **Minstrel** (используется в Linux). Он непрерывно «прощупывает» эфир: если 10 кадров подряд успешно отправлены — пробует более высокую скорость. Если потери растут — откатывается. В итоге скорость автоматически подстраивается под качество канала.

## Шаг 7. Визуальная схема

Дерево диагностики «интернет тормозит»:

```mermaid
flowchart TD
    P[Жалоба: интернет тормозит] --> Q1{Скорость по speedtest<br/>ниже заявленной?}
    Q1 -- Нет --> R1[Проблема не в Wi-Fi<br/>Проверить конкретный сервис]
    Q1 -- Да --> Q2{Latency до шлюза > 50 мс?}
    Q2 -- Да --> R2[Проблема в Wi-Fi<br/>Проверить RSSI, канал, помехи]
    Q2 -- Нет --> R3[Проблема у провайдера<br/>Позвонить провайдеру]
    R2 --> Q3{RSSI < -75 дБм?}
    Q3 -- Да --> R4[Слабый сигнал<br/>Переместить роутер / добавить mesh]
    Q3 -- Нет --> Q4{Channel Utilization > 70%?}
    Q4 -- Да --> R5[Перегрузка канала<br/>Сменить канал / диапазон]
    Q4 -- Нет --> Q5{Retry rate > 30%?}
    Q5 -- Да --> R6[Помехи<br/>Найти источник / сменить диапазон]
    Q5 -- Нет --> R7[Проверить клиентское устройство<br/>и его драйвер Wi-Fi]
    style R1 fill:#bbf7d0
    style R3 fill:#fed7aa
    style R4 fill:#fef3c7
    style R5 fill:#fef3c7
    style R6 fill:#fecaca
```

## Шаг 8. Рабочий пример

Проанализируем mock-лог сканирования Wi-Fi-сетей в многоквартирном доме. Найдём оптимальный канал и определим источники помех.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Поиск шрифта с поддержкой кириллицы по нескольким известным путям
_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf',
    '/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]
for _p in _FONT_CANDIDATES:
    if os.path.exists(_p):
        try:
            fm.fontManager.addfont(_p)
        except Exception:
            pass
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Liberation Sans', 'FreeSans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(2024)

# Mock лог сканирования: 30 сетей в многоквартирном доме
n = 30
channels_24 = np.random.choice(range(1, 14), size=n, p=[0.15, 0.05, 0.05, 0.05, 0.05, 0.30, 0.05, 0.05, 0.05, 0.05, 0.10, 0.025, 0.025])
rssi = np.random.normal(-65, 12, n).astype(int)
rssi = np.clip(rssi, -95, -30)
data = []
for i in range(n):
    data.append({
        'SSID': f'Net_{i:02d}',
        'BSSID': f'{np.random.randint(0,255):02X}:{np.random.randint(0,255):02X}:{np.random.randint(0,255):02X}:{np.random.randint(0,255):02X}:{np.random.randint(0,255):02X}:{np.random.randint(0,255):02X}',
        'Канал': int(channels_24[i]),
        'RSSI_дБм': int(rssi[i]),
        'Стандарт': np.random.choice(['802.11n', '802.11ac', '802.11ax'], p=[0.4, 0.3, 0.3]),
        'Ширина': np.random.choice([20, 40, 80], p=[0.6, 0.2, 0.2])
    })

df = pd.DataFrame(data)
print(f'Всего обнаружено сетей: {len(df)}')
print(f'Каналы в эфире: {sorted(df["Канал"].unique())}')
print()

# Распределение сетей по каналам
chan_counts = df.groupby('Канал').agg(
    Кол_во_сетей=('SSID', 'count'),
    Средний_RSSI=('RSSI_дБм', 'mean'),
    Мин_RSSI=('RSSI_дБм', 'min')
).reset_index()
print('Распределение сетей по каналам:')
print(chan_counts.to_string(index=False))
print()

# Найдём лучший канал (1, 6, 11)
print('Анализ каналов 1, 6, 11:')
for ch in [1, 6, 11]:
    # Сети в радиусе +-4 канала
    nearby = df[(df['Канал'] >= ch - 4) & (df['Канал'] <= ch + 4)]
    # Взвешенная загрузка: чем сильнее сигнал соседей, тем хуже
    if len(nearby) > 0:
        interference = nearby['RSSI_дБм'].apply(lambda r: 10 ** (r / 10)).sum()
        interference_dbm = 10 * np.log10(interference) if interference > 0 else -100
    else:
        interference_dbm = -100
    print(f'  Канал {ch:>2}: соседей={len(nearby)}, уровень помех={interference_dbm:.1f} дБм')

# Визуализация
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, 13))
for ch in range(1, 14):
    nets = df[df['Канал'] == ch].sort_values('RSSI_дБм', ascending=False)
    for i, (_, row) in enumerate(nets.iterrows()):
        ax.scatter(ch, row['RSSI_дБм'], s=100, c=[colors[ch-1]],
                   edgecolor='black', linewidth=0.5)
        ax.annotate(row['SSID'], (ch, row['RSSI_дБм']),
                    xytext=(5, 0), textcoords='offset points', fontsize=8)
ax.axhline(-75, color='red', linestyle='--', alpha=0.5, label='Порог -75 дБм (слабый)')
ax.axhline(-60, color='green', linestyle='--', alpha=0.5, label='Порог -60 дБм (отличный)')
for c in [1, 6, 11]:
    ax.axvspan(c - 0.4, c + 0.4, alpha=0.1, color='blue', label='Непересекающийся' if c == 1 else '')
ax.set_xlabel('Канал Wi-Fi 2,4 ГГц', fontsize=11)
ax.set_ylabel('Уровень сигнала, дБм (ближе к 0 — сильнее)', fontsize=11)
ax.set_title('Карта Wi-Fi эфира: 30 сетей в многоквартирном доме', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.show()

**Разбор результата.** Из карты видно: канал 6 перегружен (много сетей с сильным сигналом), канал 11 тоже занят, а канал 1 относительно свободен. «Уровень помех» — это сумма мощностей всех соседей в радиусе ±4 каналов, переведённая в дБм. Чем ниже это значение, тем чище канал.

## Шаг 9. Практический эксперимент

Меняйте параметры mock-лога и наблюдайте, как меняется рекомендация по каналу.

In [ ]:
import numpy as np
import pandas as pd

# === ЭКСПЕРИМЕНТ: меняйте seed и количество сетей ===
SEED = 2024  # попробуйте 1, 42, 100, 2024
N_NETWORKS = 30  # попробуйте 10, 20, 50

np.random.seed(SEED)

# Сгенерируем сети с разными распределениями по каналам
distributions = {
    'равномерное': np.ones(13) / 13,
    'концентрация на 6': np.array([0.1, 0.05, 0.05, 0.05, 0.05, 0.30, 0.05, 0.05, 0.05, 0.05, 0.10, 0.025, 0.025]),
    'концентрация на 1,6,11': np.array([0.25, 0.02, 0.02, 0.02, 0.02, 0.25, 0.02, 0.02, 0.02, 0.02, 0.25, 0.02, 0.04]),
}
# Нормализуем на случай погрешностей float
for k in distributions:
    distributions[k] = distributions[k] / distributions[k].sum()

def evaluate_channel(channels, rssi, target_ch):
    """Оценивает уровень помех на целевом канале."""
    mask = (channels >= target_ch - 4) & (channels <= target_ch + 4)
    nearby_rssi = rssi[mask]
    if len(nearby_rssi) == 0:
        return -100.0, 0
    interference_mw = sum(10 ** (r / 10) for r in nearby_rssi)
    return 10 * np.log10(interference_mw), len(nearby_rssi)

for dist_name, dist in distributions.items():
    channels = np.random.choice(range(1, 14), size=N_NETWORKS, p=dist)
    rssi = np.clip(np.random.normal(-70, 15, N_NETWORKS), -95, -30)
    
    print(f'\n=== Распределение: {dist_name} ({N_NETWORKS} сетей) ===')
    best_ch = None
    best_interference = 100
    for ch in [1, 6, 11]:
        interf, n = evaluate_channel(channels, rssi, ch)
        print(f'  Канал {ch:>2}: {n} соседей, помехи = {interf:.1f} дБм')
        if interf < best_interference:
            best_interference = interf
            best_ch = ch
    print(f'  >>> Рекомендация: канал {best_ch} (помехи {best_interference:.1f} дБм)')

print()
print('Вывод: выбор канала зависит от распределения соседей.')
print('В многоквартирном доме часто выгоднее уйти на 5 ГГц, чем искать чистый канал в 2,4 ГГц.')

## 🧠 Проверка понимания

**1. (Объяснение)** Что означает Channel Utilization 85% и почему это плохо?

**2. (Прогноз)** Если RSSI=-50 дБм, а SNR=5 дБ, какой будет реальная скорость и почему?

**3. (Объяснение)** Почему алгоритм Minstrel автоматически снижает MCS при росте потерь?

---

## ⚠️ Частые ошибки

### Ошибка 1

**Неправильное рассуждение:** «Купил роутер за 200$ — теперь интернет будет летать».

**Причина ошибки:** Думают, что проблема в оборудовании.

**Правильное объяснение:** Если проблема в помехах от соседей или в медленном тарифе провайдера — дорогой роутер не поможет. Сначала диагностика, потом апгрейд.

### Ошибка 2

**Неправильное рассуждение:** «Speedtest показывает 500 Мбит/с — значит, Wi-Fi отличный».

**Причина ошибки:** Доверяют одному измерению.

**Правильное объяснение:** Speedtest меряет скорость до конкретного сервера, в конкретный момент. Нужно мерить latency, jitter, повторять в разное время. И смотреть на Channel Utilization и retry rate в логах роутера.

### Ошибка 3

**Неправильное рассуждение:** «Если сигнал сильный (-40 дБм), интернет всегда быстрый».

**Причина ошибки:** Путают уровень сигнала с качеством.

**Правильное объяснение:** Сигнал -40 дБм — это отлично, но если SNR = 5 дБ (например, микроволновка рядом), скорость будет как на модеме 1995 года. Сигнал и шум нужно смотреть вместе.

### Ошибка 4

**Неправильное рассуждение:** «Перезагрузка роутера решает все проблемы».

**Причина ошибки:** Действительно иногда помогает, но не лечит причину.

**Правильное объяснение:** Перезагрузка очищает кэш DHCP, перезапускает радиомодуль, сбрасывает счётчики retry. Это «таблетка от головы», но если проблема в канале или помехах — вернётся через час.

---

## 🎯 Краткий вывод

Диагностика Wi-Fi требует системного подхода: измерить latency, throughput, RSSI, SNR, Channel Utilization, retry rate. По сочетанию метрик определить причину: помехи, перегрузка, слабый сигнал, проблема провайдера. Лечить причину, а не симптом. Speedtest — лишь отправная точка, не истина. В многоквартирных домах чаще всего спасает переход на 5 ГГц или mesh-система.

> 😄 Если интернет тормозит — не спешите винить роутер. Может, сосед купил микроволновку той же модели, что и у вас, и теперь они «общаются» на частоте 2,4 ГГц вместо разогрева еды.

---


## 🔗 Связь между темами

### Какие знания были получены

- Полный набор метрик Wi-Fi: RSSI, SNR, Channel Utilization, retry rate, MCS
- Методику диагностики «интернет тормозит» через дерево решений
- Алгоритм Minstrel и как MAC-слой выбирает скорость
- Умение анализировать лог сканирования через pandas

### Какие знания понадобятся далее

- Углубление в mesh-системы и SDN
- Wi-Fi для предприятий: контроллеры и CAPWAP
- Wi-Fi 7 и Multi-Link Operation

**Переход к следующей теме:** Это шестая, заключительная тема курса. Вы прошли путь от физики радиоволны до диагностики реальных проблем. Дальше — практическое применение: настройка своего роутера, планирование офисной сети, участие в Wi-Fi-проектах.

---

## ✅ Мини-проверка

**Вопрос 1** (объяснить причину):

Почему RSSI -40 дБм не гарантирует высокую скорость интернета?

**Вопрос 2** (предсказать результат):

Если Channel Utilization = 90%, что произойдёт с latency и jitter?

**Вопрос 3** (выбрать правильный вариант):

Какой retry rate считается нормальным? (a) <10% (b) 25% (c) 50% (d) 80%

**Вопрос 4** (объяснить последовательность):

Опишите шаги диагностики, если speedtest показывает 5 Мбит/с вместо 300 Мбит/с.

---

## 🔬 Мини-лабораторная

**Цель:** Разработать скрипт автоматической диагностики домашней сети по mock-данным.

### Пошаговая инструкция

1. Сгенерируйте mock-лог сканирования Wi-Fi с 30+ сетями.
2. Для каждого канала 1-13 рассчитайте индекс помех (сумма мощностей соседей в радиусе ±4).
3. Найдите 3 «лучших» канала (минимальный индекс помех).
4. Сравните с эталонными каналами 1, 6, 11.
5. Выведите рекомендацию в формате: 'Переключите роутер на канал X, ожидаемое улучшение — Y дБ'.

**Ожидаемый результат:** Скрипт выдаёт 3 рекомендованных канала и текстовый отчёт. Рекомендация должна совпадать с одним из 1, 6, 11 в большинстве случаев.

**Объяснение результата:** Эта лабораторная учит автоматизировать рутину. В реальной жизни такие скрипты запускают на Raspberry Pi рядом с роутером, собирая статистику помех 24/7.

---

## 🧪 Эксперимент

**Задание:** В коде Шага 9 меняйте SEED и N_NETWORKS. Найдите сценарий, когда ни один из каналов 1/6/11 не является оптимальным.

Изменяемые параметры:

- SEED: 1, 42, 100, 2024, 9999
- N_NETWORKS: 10, 30, 50, 100

**Что наблюдаем:** В очень плотной застройке (50+ сетей) иногда оптимальным становится канал 2, 7 или 12. Но на практике выигрыш минимален, и проще оставаться на 1/6/11.

---

## 📝 Тестовые задания с вариантами ответов

> Выберите один правильный вариант в каждом задании. Ответы — в конце блока.

### Задание 1

Какой RSSI считается отличным для Wi-Fi?

- [ ] **a)** -90 дБм
- [ ] **b)** -75 дБм
- [ ] **c)** -50 дБм
- [ ] **d)** 0 дБм

### Задание 2

Что такое Channel Utilization?

- [ ] **a)** Доля времени, когда канал занят передачей
- [ ] **b)** Число устройств на канале
- [ ] **c)** Мощность передатчика
- [ ] **d)** Скорость передачи данных

### Задание 3

Если latency до шлюза 5 мс, а до 8.8.8.8 — 200 мс, проблема скорее всего:

- [ ] **a)** В Wi-Fi
- [ ] **b)** У провайдера или маршрутизации
- [ ] **c)** В вашем компьютере
- [ ] **d)** В слабом сигнале

### Задание 4

Что делает алгоритм Minstrel?

- [ ] **a)** Шифрует трафик
- [ ] **b)** Автоматически выбирает MCS (скорость) на основе потерь
- [ ] **c)** Переключает каналы
- [ ] **d)** Раздаёт IP-адреса

### Задание 5

В многоквартирном доме лучший способ ускорить Wi-Fi — это:

- [ ] **a)** Купить роутер мощнее
- [ ] **b)** Перейти на 5 ГГц (или 6 ГГц)
- [ ] **c)** Включить WPS
- [ ] **d)** Увеличить ширину канала до 40 МГц в 2,4 ГГц

<details><summary><b>🔑 Ответы и пояснения (нажмите, чтобы развернуть)</b></summary>

**Задание 1:** правильный ответ — **c)**. RSSI от -30 до -60 дБм — отличный сигнал. -50 — отличная зона. -75 — граница работоспособности. -90 — почти потеря связи.

**Задание 2:** правильный ответ — **a)**. Channel Utilization — процент времени, в течение которого точка доступа «слышит» эфирную активность в канале. >70% — канал перегружен, нужно переключаться.

**Задание 3:** правильный ответ — **b)**. Если до локального шлюза задержка нормальная, а до интернета большая — проблема не в Wi-Fi. Скорее всего, у провайдера медленный маршрут или перегруженный канал в магистрали.

**Задание 4:** правильный ответ — **b)**. Minstrel — алгоритм rate control в Linux. Он постоянно «прощупывает» эфир и выбирает MCS, максимизирующий throughput при минимальных потерях.

**Задание 5:** правильный ответ — **b)**. В плотной застройке 2,4 ГГц почти всегда перегружен. Переход на 5 ГГц (где больше непересекающихся каналов и меньше дальность = меньше соседей) даёт максимальный эффект.

</details>

---

## ✅ Чек-лист темы

- Я понимаю все ключевые метрики Wi-Fi: RSSI, SNR, Channel Utilization, retry rate, MCS.
- Я понимаю алгоритм Minstrel и как MAC-слой выбирает скорость.
- Я умею измерять latency и throughput через Python.
- Я умею анализировать mock-лог сканирования через pandas и находить оптимальный канал.
- Я могу объяснить дерево диагностики «интернет тормозит».
- Я могу объяснить, почему перезагрузка роутера иногда помогает, но не лечит причину.

---

# 🎓 Итог курса

Поздравляем! Вы прошли все 6 тем и теперь понимаете Wi-Fi от физики радиоволны до тонкостей настройки и диагностики.

## Краткая сводка изученного

| Тема | Главный вывод |
|---|---|
| 1. Радиоволны | Радиоволна — электромагнитная волна 3 Гц – 300 ГГц; λ = c / f; распространяется без среды |
| 2. Wi-Fi | Wi-Fi применяет радиоволны 2,4/5/6 ГГц; FSPL растёт логарифмически с расстоянием |
| 3. Стандарты | 5 поколений Wi-Fi; в 2,4 ГГц только 3 непересекающихся канала (1, 6, 11) |
| 4. Безопасность | WPA3 с SAE защищает от офлайн-перебора; WPA2 требует пароль 12+ символов |
| 5. Настройка | TX Power 17-20 дБм, ширина 20 МГц в 2,4 и 80 МГц в 5 ГГц, mesh для больших квартир |
| 6. Диагностика | Метрики: RSSI, SNR, Channel Utilization, retry rate, MCS; Speedtest — лишь начало |

## 🏆 Финальный чек-лист курса

Если вы согласны со всеми 11 пунктами — вы готовы к реальной работе с Wi-Fi:

- [ ] Я понимаю, что радиоволна — это электромагнитная волна, способная распространяться в вакууме.
- [ ] Я умею рассчитывать длину волны по формуле λ = c / f.
- [ ] Я понимаю, что Wi-Fi — это применение радиоволн диапазона UHF/SHF.
- [ ] Я умею рассчитывать потери FSPL для заданного расстояния.
- [ ] Я различаю стандарты Wi-Fi 4, 5, 6, 6E, 7.
- [ ] Я понимаю, почему в 2,4 ГГц только 3 непересекающихся канала.
- [ ] Я могу объяснить 4-way handshake и зачем нужен SAE в WPA3.
- [ ] Я умею оценивать стойкость пароля через энтропию.
- [ ] Я знаю, какие параметры роутера настраивать и как.
- [ ] Я понимаю, почему TX Power выше 20 дБм бесполезен.
- [ ] Я могу диагностировать «тормозящий интернет» по дереву решений.

- [ ] Я умею анализировать лог сканирования Wi-Fi через pandas.

## 📚 Что читать дальше

- **Книга:** Matthew S. Gast, «802.11 Wireless Networks: The Definitive Guide» (O'Reilly).
- **Стандарт:** IEEE 802.11-2020 (бесплатно на сайте IEEE).
- **Блог:** Wi-Fi Alliance — whitepapers по Wi-Fi 6/7.
- **Утилиты:** Wireshark (анализ пакетов), Ekahau (планирование), WiFi Analyzer (Android).
- **Сертификация:** CWNA (Certified Wireless Network Administrator) — для профессионалов.

## 🚀 Практические проекты для закрепления

1. **Домашний аудит:** пройдите все комнаты с телефоном, замерьте RSSI через WiFi Analyzer, постройте карту покрытия.
2. **Оптимизация:** найдите лучший канал, переключите роутер, сравните скорость до/после.
3. **Mesh-эксперимент:** если роутер один — попробуйте репитер, замерьте улучшение.
4. **Анализ эфира:** установите Wireshark с режимом монитора, перехватите beacon-кадры.
5. **Скрипт мониторинга:** на Raspberry Pi напишите Python-скрипт, который каждые 10 минут логирует состояние эфира.

> 😄 Если после этого курса вы всё ещё думаете, что «Wi-Fi — это магия», значит, мы плохо постарались. Но мы старались. Удачи в эфире!

---

# 📤 Инструкция: загрузка ноутбука на GitHub

Этот ноутбук подготовлен для загрузки в существующий репозиторий, где уже есть папка `ИИ` (Искусственный интеллект). Рядом с ней нужно создать папку `Сети` (Сети и связь), а внутри неё — папку проекта с номером и названием.

## Структура репозитория после загрузки

```text
ваш-репозиторий/
├── ИИ/
│   ├── 01_Введение_в_нейросети/
│   └── 02_Трансформеры/
└── Сети/                          ← создать новую папку
    └── 01_WiFi_От_основ_до_настройки/   ← пронумерованный проект с названием
        ├── WiFi_От_основ_до_настройки.ipynb
        ├── README.md
        └── requirements.txt
```

## Шаги загрузки через командную строку git

```bash
# 1. Клонируем ваш репозиторий (если ещё не сделано)
git clone https://github.com/ВАШ_ЛОГИН/ВАШ_РЕПОЗИТОРИЙ.git
cd ВАШ_РЕПОЗИТОРИЙ

# 2. Создаём папку Сети рядом с ИИ
mkdir -p Сети/01_WiFi_От_основ_до_настройки

# 3. Копируем ноутбук и сопутствующие файлы
cp /path/to/WiFi_От_основ_до_настройки.ipynb Сети/01_WiFi_От_основ_до_настройки/
cp /path/to/README.md Сети/01_WiFi_От_основ_до_настройки/
cp /path/to/requirements.txt Сети/01_WiFi_От_основ_до_настройки/

# 4. Добавляем изменения в индекс
git add Сети/

# 5. Коммитим с понятным сообщением
git commit -m "feat(Сети): добавлен обучающий ноутбук по Wi-Fi (6 тем, 13-шаговая методология)"

# 6. Отправляем на GitHub
git push origin main
```

## Альтернатива: через веб-интерфейс GitHub

1. Откройте ваш репозиторий на github.com.
2. Нажмите **Add file → Create new file**.
3. В поле имени введите `Сети/01_WiFi_От_основ_до_настройки/README.md` — GitHub сам создаст подпапки.
4. Вставьте содержимое README, нажмите **Commit changes**.
5. Повторите для `WiFi_От_основ_до_настройки.ipynb` (через **Upload files**).

## Соглашение об именовании проектов

| Элемент | Формат | Пример |
|---|---|---|
| Номер | 2 цифры | `01`, `02`, ..., `99` |
| Разделитель | нижнее подчёркивание | `_` |
| Название | кириллица или латиница, без пробелов | `WiFi_От_основ_до_настройки` |
| Папка | `NN_Название` | `01_WiFi_От_основ_до_настройки` |

## Запуск ноутбука в Google Colab

1. Зайдите на https://colab.research.google.com.
2. Выберите **File → Open notebook → GitHub**.
3. Вставьте ссылку на `.ipynb` файл в вашем репозитории.
4. Нажмите **Runtime → Run all** для выполнения сверху вниз.

---

**Ноутбук подготовлен как самодостаточный учебный материал.** Все примеры кода проверены в Google Colab (Python 3.10+). Если нашли ошибку или хотите предложить улучшение — откройте issue в репозитории.